# Эксперимент 1: обычная LLaDA-8B-Instruct (HMM не используется) - блокнот-пульт

Проект "Токенные зависимости в диффузионной генерации" (абляции метода CoDD, arXiv 2603.00045). Файл: `exp1_llada_default.ipynb`.
Режим: `use_pc=False` - слова-кандидаты и уверенность (значит, и порядок раскрытия) считаются по логитам нейросети, HMM не участвует.

**Схема "пульт + процесс".** Ядро этого блокнота (DataSphere `mlkernel`) - старое окружение (torch 2.0.1), поэтому здесь используется **только стандартная библиотека**. Весь тяжёлый код лежит в `exp1_core.py`, который блокнот записывает на диск и запускает **отдельным процессом** из Python текущего ядра (Python 3.10, torch 2.9.1+cu126, transformers 4.57.3, pyjuice 2.4.1, lm_eval из CoDD). Процесс живёт независимо от ядра: если ядро перезапустится, расчёт продолжится, а ячейка мониторинга снова к нему подключится.

> ## ЗАМЕТНЫЕ ПОМЕТКИ (прочитать до запуска)
> 1. **В блокнотах 2-5 (с HMM) заглушка pyjuice ЗАПРЕЩЕНА.** В exp1 путь HMM не вызывается, поэтому в аварийном случае (pyjuice не импортируется в venv) заглушка допустима (`ALLOW_PYJUICE_STUB`, по умолчанию выключено), но тогда будет громкое предупреждение и запись `pyjuice: stub` в `env.json`.
> 2. **`HMM_CKPT` изменён по решению автора проекта**: в исходном задании было `il18/llada-math-pc`, здесь `il18/llada-gsm-pc`. В exp1 не используется, константа общая для пяти блокнотов.
> 3. **`GEN_LENGTH = 512`** (единственная константа длины генерации). В статье (приложение B) и в `eval/harness.py` - 512, в README-примере и `example.py` - 256. Менять только по решению автора проекта.
> 4. Блокнот **не запускался** при подготовке: проверены только синтаксис (`ast.parse`) и структура (`nbformat.validate`). Всё непроверенное помечено словами **НЕ ПРОВЕРЕНО**.
> 5. GPU L4 (compute capability 8.9) поддерживает bf16 аппаратно, поэтому `DTYPE = "bfloat16"`. Для остальных четырёх блокнотов сопоставимость требует того же dtype.
> 6. **Остановка машины**: блокнот не может остановить ВМ. По окончании работы остановите JupyterLab и ВМ в интерфейсе DataSphere (иначе продолжится тарификация GPU).

Текущий прогон на подвыборке: GSM8K test, первые 300 задач; GEN_LENGTH=512, BLOCK_LENGTH=32, margin, 128 шагов, SEEDS=[0]. Производная seed совпадает с exp3: SHA-256 строки exp3:{seed}:{task_id}, первые 4 байта little-endian. Метрика и алгоритм эксперимента не изменены. Папки нового прогона имеют суффикс _margin128_seed0_gen512_n300. GPU-запуск при этой правке НЕ ПРОВЕРЕНО: выполнены только статические проверки.


Настройки запуска перенесены из exp3: REPO_ROOT=/home/jupyter/project/CoDD-ablations,
STORAGE_ROOT=/home/jupyter/project, ASSETS_DIR=../exp3-assets относительно REPO_ROOT.
Используются ASSETS_DIR/model, ASSETS_DIR/hmm/model.jpc и ASSETS_DIR/gsm8k
(DatasetDict/save_to_disk); для другого расположения измените ASSETS_DIR в конфиге.
При ASSETS_DIR=None доступны ручные пути MODEL_LOCAL_DIR, HMM_LOCAL_PATH и DATASET_LOCAL_DIR.
Python процесса совпадает с Python текущего ядра; отдельный venv не требуется,
user-site зависимости из .local доступны. Сеть и автоустановка отключены.
Поддержка локальных файлов, готового manifest и устройства CUDA согласована с exp3;
формулы генерации, метрика и выбранная конфигурация margin/128/seed0/512/300 сохранены.
При текущей правке выполнены только статические проверки. GPU-запуск НЕ ПРОВЕРЕНО.


In [ ]:
# ===================== 0. КОНФИГ (первая ячейка с кодом; только стандартная библиотека) =====================
import os
import sys
import json
import time
import signal
import hashlib
import datetime
import subprocess
from pathlib import Path

# ---------- ОБЩИЕ ПАРАМЕТРЫ (одинаковы у всех пяти блокнотов, менять нельзя) ----------
MODEL = "GSAI-ML/LLaDA-8B-Instruct"
HMM_CKPT = "il18/llada-gsm-pc"   # ИЗМЕНЕНО по решению автора проекта (в задании было "il18/llada-math-pc"). В exp1 НЕ используется.
DATASET = "gsm8k"
DATASET_SPLIT = "test"
N_TASKS = 300                    # первые 300 задач test-выборки в порядке датасета
GEN_LENGTH = 512                 # ЕДИНСТВЕННАЯ константа длины генерации (256 или 512). Статья и eval/harness.py: 512; README-пример и example.py: 256.
BLOCK_LENGTH = 32
TEMPERATURE = 0.0                # жадный выбор, как в статье
NUM_STEPS_LIST = [128]
REMASKING_LIST = ["margin"]
PC_TEMPERATURE = 0.2             # из CLI-примера README. В exp1 не используется.
PC_FRAC = 0.5                    # из CLI-примера README. В exp1 не используется.
REVERSE_FRAC = False             # дефолт eval.py. В exp1 не используется.
SEEDS = [0]                    # один повтор; task_seed согласован с exp3

# ---------- ПАРАМЕТРЫ ЭКСПЕРИМЕНТА 1 ----------
EXP_NAME = "exp1_llada_default"
USE_PC = False                   # обычная LLaDA: HMM не используется
SEED_USED = 0                    # номер повтора; перед задачей вычисляется производный task_seed
SYSTEM_INSTRUCTION = "You are a helpful assistant."   # eval.py: для gsm8k именно этот system prompt
NUM_FEWSHOT = 0                  # eval.py: --num_shot по умолчанию 0
TASK_NAME = "gsm8k"              # имя задачи в lm_eval
MASK_ID = 126336
DTYPE = "bfloat16"               # "bfloat16" | "float16". L4 (capability 8.9) поддерживает bf16 аппаратно.
ATTN_IMPLEMENTATION = "eager"    # eval.py: attn_implementation="eager"
N_BOOTSTRAP = 2000              # ресэмплов для доверительного интервала (по задачам)
BOOTSTRAP_SEED = 20261023           # seed ТОЛЬКО для bootstrap (генерация случайности не использует)

# ---------- НАСТРОЙКИ DATASPHERE ИЗ EXP3 ----------
# Относительный ASSETS_DIR отсчитывается от REPO_ROOT, как после %cd .. в exp3.
REPO_ROOT = Path("/home/jupyter/project/CoDD-ablations")
STORAGE_ROOT = Path("/home/jupyter/project")
ASSETS_DIR = Path(r"../exp3-assets")
OFFLINE = True
DEVICE = "cuda:0"
MODEL_LOCAL_DIR = None
HMM_LOCAL_PATH = None
DATASET_LOCAL_DIR = None
HF_CACHE_DIR = None

REPO_ROOT = REPO_ROOT.expanduser().resolve()
STORAGE_ROOT = STORAGE_ROOT.expanduser().resolve()
if ASSETS_DIR is not None:
    assert all(v is None for v in (MODEL_LOCAL_DIR, HMM_LOCAL_PATH, DATASET_LOCAL_DIR)), (
        "Выберите ASSETS_DIR либо ручные пути, как в exp3.")
    _assets = Path(ASSETS_DIR).expanduser()
    ASSETS_DIR = (_assets if _assets.is_absolute() else REPO_ROOT / _assets).resolve()
    MODEL_PATH = str(ASSETS_DIR / "model")
    DATA_DIR = str(ASSETS_DIR / "gsm8k")
    HMM_FILE = str(ASSETS_DIR / "hmm" / "model.jpc")
    ASSETS_MANIFEST = str(ASSETS_DIR / "download_manifest.json")
else:
    assert MODEL_LOCAL_DIR is not None and DATASET_LOCAL_DIR is not None, (
        "Укажите MODEL_LOCAL_DIR и DATASET_LOCAL_DIR.")
    assert EXP_NAME == "exp1_llada_default" or HMM_LOCAL_PATH is not None, "Укажите HMM_LOCAL_PATH."
    MODEL_PATH = str(Path(MODEL_LOCAL_DIR).expanduser().resolve())
    DATA_DIR = str(Path(DATASET_LOCAL_DIR).expanduser().resolve())
    HMM_FILE = str(Path(HMM_LOCAL_PATH).expanduser().resolve()) if HMM_LOCAL_PATH is not None else ""
    ASSETS_MANIFEST = ""
assert OFFLINE is True, "DataSphere работает без внешней сети; используйте локальные файлы."
CODD_REPO_DIR = str(REPO_ROOT)
PROJECT_DIR = str(STORAGE_ROOT)
PROJECT_DIR_SOURCE = "STORAGE_ROOT, как в exp3"
HMM_DIR = str(Path(HMM_FILE).parent) if HMM_FILE else ""
HMM_MANIFEST = ASSETS_MANIFEST
# SHA-256 gsm-PC из проверенных артефактов exp3; ready-manifest содержит ревизии, а не хэш файла.
EXPECTED_HMM_SHA256 = "0c8903455fd8cf592e8c8d5489230eb0d343c6bc6adae3208c533d09dc51c25f"
# Используем тот же Python и user-site пакеты, что и рабочее ядро exp3.
VENV_PYTHON = sys.executable  # имя оставлено для совместимости с пультом; отдельный venv не нужен
PROCESS_PYTHON_FLAGS = []    # -I скрывает установленные в .local зависимости
WORK_DIR = str(STORAGE_ROOT / "exp1_run_margin128_seed0_gen512_n300")
OUT_DIR = str(STORAGE_ROOT / "results" / (EXP_NAME + "_margin128_seed0_gen512_n300"))
EXP1_RESULTS_DIR = str(STORAGE_ROOT / "results" / "exp1_llada_default_margin128_seed0_gen512_n300")
EXP5_RESULTS_DIR = str(STORAGE_ROOT / "results" / "exp5_codd_full_margin128_seed0_gen512_n300")
ALLOW_PYJUICE_STUB = False
ALLOW_MODEL_CODE_MISMATCH = False  # True: продолжить, даже если modeling_llada.py не совпал по sha256 с файлом рядом с весами
STRICT_VERSIONS = False          # True: расхождение версий с ожидаемыми - ошибка (по умолчанию предупреждение)
MIN_GPU_TOTAL_GB = 20            # ГиБ (у карты "на 24 ГБ" реально ~22.5 ГиБ)
MIN_GPU_FREE_GB = 20
RUN_LIMIT = None                 # None = все N_TASKS; число = только первые RUN_LIMIT задач в ПОЛНОМ прогоне (tasks.json остаётся из 300 задач)
MAX_RUNTIME_MIN = None           # None = без ограничения; число минут: процесс после ближайшей завершённой задачи корректно остановится
REQUIRE_SMOKE = True             # полный прогон требует успешного дымового теста (smoke.json)
MONITOR_INTERVAL_S = 30          # период обновления мониторинга (ядро остаётся занятым, чтобы ВМ не считалась простаивающей)
MONITOR_TAIL_LINES = 12

# ---------- ПРОВЕРКИ ПАРАМЕТРОВ ----------
assert MODEL == "GSAI-ML/LLaDA-8B-Instruct", "MODEL - общий параметр, менять нельзя"
assert HMM_CKPT == "il18/llada-gsm-pc", "HMM_CKPT - общий параметр (см. пометку 2 в начале блокнота)"
assert (DATASET, DATASET_SPLIT, N_TASKS) == ("gsm8k", "test", 300), "DATASET/DATASET_SPLIT/N_TASKS - общие параметры"
assert GEN_LENGTH == 512, "GEN_LENGTH=512 для выбранного полного прогона"
assert BLOCK_LENGTH == 32 and TEMPERATURE == 0.0, "BLOCK_LENGTH/TEMPERATURE - общие параметры"
assert NUM_STEPS_LIST == [128] and REMASKING_LIST == ["margin"], "NUM_STEPS_LIST/REMASKING_LIST - общие параметры"
assert (PC_TEMPERATURE, PC_FRAC, REVERSE_FRAC) == (0.2, 0.5, False), "PC_* - общие параметры"
assert SEEDS == [0], "SEEDS - номер повтора, согласованный с другими экспериментами"
assert SEED_USED == SEEDS[0] == 0
assert USE_PC is False, "exp1 - это эксперимент БЕЗ HMM"
assert DTYPE in ("bfloat16", "float16"), "DTYPE: 'bfloat16' или 'float16'"
assert RUN_LIMIT is None or (isinstance(RUN_LIMIT, int) and 1 <= RUN_LIMIT <= N_TASKS), "RUN_LIMIT: None или целое от 1 до N_TASKS"
assert MAX_RUNTIME_MIN is None or (isinstance(MAX_RUNTIME_MIN, (int, float)) and MAX_RUNTIME_MIN > 0), "MAX_RUNTIME_MIN: None или число минут > 0"
for _gl in sorted({256, 512, GEN_LENGTH}):
    assert _gl % BLOCK_LENGTH == 0, "GEN_LENGTH=%d не делится на BLOCK_LENGTH=%d" % (_gl, BLOCK_LENGTH)
    for _ns in NUM_STEPS_LIST:
        assert _ns % (_gl // BLOCK_LENGTH) == 0, "num_steps=%d не делится на число блоков (GEN_LENGTH=%d)" % (_ns, _gl)
NUM_BLOCKS = GEN_LENGTH // BLOCK_LENGTH
print("GEN_LENGTH=%d: блоков %d; токенов за шаг: %s" % (GEN_LENGTH, NUM_BLOCKS, ", ".join("%d шагов -> %g" % (ns, GEN_LENGTH / float(ns)) for ns in NUM_STEPS_LIST)))


def sha256_json(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")).hexdigest()


COMMON_PARAMS = {
    "MODEL": MODEL, "HMM_CKPT": HMM_CKPT, "DATASET": DATASET, "DATASET_SPLIT": DATASET_SPLIT, "N_TASKS": N_TASKS,
    "GEN_LENGTH": GEN_LENGTH, "BLOCK_LENGTH": BLOCK_LENGTH, "TEMPERATURE": TEMPERATURE,
    "NUM_STEPS_LIST": NUM_STEPS_LIST, "REMASKING_LIST": REMASKING_LIST,
    "PC_TEMPERATURE": PC_TEMPERATURE, "PC_FRAC": PC_FRAC, "REVERSE_FRAC": REVERSE_FRAC, "SEEDS": SEEDS,
}
print("sha256 общих параметров:", sha256_json(COMMON_PARAMS))
assert sha256_json(COMMON_PARAMS) == "d9fc8dc825bdeecb34b0c190d445c02860b6a75b45a4a52c95882a3effb9f4ef", "Хэш выбранной сетки не совпал"

# Всё, что нужно процессу (читается из config.json). Ключи ОБЯЗАТЕЛЬНЫ: exp1_core.py проверяет их наличие.
CORE_CONFIG = {
    "DEVICE": DEVICE, "OFFLINE": OFFLINE, "ASSETS_MANIFEST": ASSETS_MANIFEST,
    "EXPECTED_HMM_SHA256": EXPECTED_HMM_SHA256, "HMM_LOCAL_PATH": HMM_FILE,
    "HF_CACHE_DIR": str(Path(HF_CACHE_DIR).expanduser().resolve()) if HF_CACHE_DIR is not None else None,
    "MODEL": MODEL, "HMM_CKPT": HMM_CKPT, "DATASET": DATASET, "DATASET_SPLIT": DATASET_SPLIT, "N_TASKS": N_TASKS,
    "GEN_LENGTH": GEN_LENGTH, "BLOCK_LENGTH": BLOCK_LENGTH, "TEMPERATURE": TEMPERATURE, "NUM_STEPS_LIST": NUM_STEPS_LIST,
    "REMASKING_LIST": REMASKING_LIST, "PC_TEMPERATURE": PC_TEMPERATURE, "PC_FRAC": PC_FRAC, "REVERSE_FRAC": REVERSE_FRAC, "SEEDS": SEEDS,
    "EXP_NAME": EXP_NAME, "USE_PC": USE_PC, "SEED_USED": SEED_USED, "SYSTEM_INSTRUCTION": SYSTEM_INSTRUCTION,
    "NUM_FEWSHOT": NUM_FEWSHOT, "TASK_NAME": TASK_NAME, "MASK_ID": MASK_ID, "DTYPE": DTYPE, "ATTN_IMPLEMENTATION": ATTN_IMPLEMENTATION,
    "N_BOOTSTRAP": N_BOOTSTRAP, "BOOTSTRAP_SEED": BOOTSTRAP_SEED, "PROJECT_DIR": PROJECT_DIR, "CODD_REPO_DIR": CODD_REPO_DIR,
    "MODEL_PATH": MODEL_PATH, "DATA_DIR": DATA_DIR, "OUT_DIR": OUT_DIR, "ALLOW_PYJUICE_STUB": ALLOW_PYJUICE_STUB,
    "ALLOW_MODEL_CODE_MISMATCH": ALLOW_MODEL_CODE_MISMATCH, "STRICT_VERSIONS": STRICT_VERSIONS, "MIN_GPU_TOTAL_GB": MIN_GPU_TOTAL_GB,
    "MIN_GPU_FREE_GB": MIN_GPU_FREE_GB, "RUN_LIMIT": RUN_LIMIT, "MAX_RUNTIME_MIN": MAX_RUNTIME_MIN, "REQUIRE_SMOKE": REQUIRE_SMOKE,
}
print("PROJECT_DIR = %s   <- %s" % (PROJECT_DIR, PROJECT_DIR_SOURCE))
print("Интерпретатор процесса:", VENV_PYTHON)
print("Веса:", MODEL_PATH)
print("Данные:", DATA_DIR)
print("Результаты:", OUT_DIR)
print("DTYPE:", DTYPE, "| GEN_LENGTH:", GEN_LENGTH, "| RUN_LIMIT:", RUN_LIMIT, "| MAX_RUNTIME_MIN:", MAX_RUNTIME_MIN)
print()
print("ПОМЕТКИ: HMM_CKPT изменён на il18/llada-gsm-pc (по решению автора); GEN_LENGTH =", GEN_LENGTH, "(статья: 512);")
print("         в блокнотах 2-5 заглушка pyjuice ЗАПРЕЩЕНА; блокнот не запускался при подготовке (только ast.parse и nbformat.validate).")

## Как это устроено и что проверено

**Порядок работы:** (1) проверка окружения venv; (2) запись `exp1_core.py` и `config.json`; (3) дымовой тест `--mode smoke` (сам запускает процесс и следит за ним); (4) полный прогон `--mode full`; (5) сводка `--mode summary`. Между шагами можно перезапускать ядро: процесс не зависит от ядра, ячейка мониторинга находит его по pid-файлу `exp1_run/exp1.pid`.

**Что делает процесс** (то же, что делали ячейки прежней версии блокнота; форматы результатов и steplog, `env.json`, `config.json`, `tasks.json`, `smoke.json` не менялись):
- проверка окружения и GPU, загрузка модели с диска (`local_files_only=True`), сверка sha256 `modeling_llada.py`/`configuration_llada.py` с файлами рядом с весами;
- задача GSM8K из `lm_eval` репозитория, но **датасет берётся из локального parquet**: конфиг задачи переопределяется (`dataset_path="parquet"`, `dataset_kwargs.data_files`), остальное (промпт, чат-шаблон, фильтры, метрика) строит и считает код `lm_eval` без изменений;
- построчная копия `llada_diffusion_generate` с журналом шагов; дымовой тест (копия == оригинал на `PlainView` и на `CoddLlada(use_pc=False)`), пик видеопамяти, прогноз времени;
- кэш после каждой задачи (steplog, затем результат, `fsync`), возобновление, `RUN_LIMIT`, `MAX_RUNTIME_MIN`, остановка по сигналу SIGTERM после завершённой задачи.

**НЕ ПРОВЕРЕНО** (блокнот и процесс не запускались):
- офлайн-загрузка GSM8K через переопределение конфига задачи `lm_eval` (`dataset_path="parquet"`): логика прочитана в `lm_eval/tasks/__init__.py` и `api/task.py`, но на реальных данных не выполнялась; поведение `datasets.load_dataset("parquet", ...)` при `HF_DATASETS_OFFLINE=1` не проверено;
- импорт и работа pyjuice 2.4.1 на Python 3.10 / torch 2.9.1+cu126 / драйвере 535 и GPU L4; совместимость torch 2.9.1+cu126 с драйвером; работа triton 3.5.1;
- совместимость transformers 4.57.3 и torch 2.9.1 с `modeling_llada.py`; `HFLM(pretrained=<CoddLlada>)`;
- что `-I`-режим venv находит editable-установку `lm_eval` и остальные пакеты;
- остановка процесса через группу процессов (`start_new_session=True` + `os.killpg`), определение живого процесса по `/proc` после перезапуска ядра, `IPython.display.clear_output` в ядре `mlkernel`, версия Python в `mlkernel`;
- побитовая воспроизводимость на другом GPU; время полного прогона (оценка печатается дымовым тестом);
- на каком коде посчитаны базовые цифры `margin` в статье: оригинальный харнесс без HMM поддерживает только `low_confidence` и `random`.

## 1. Проверка окружения (venv, отдельный процесс)

Запускается Python текущего ядра без флага -I с теми же переменными среды, что и у рабочего процесса. Выводится интерпретатор, версии, наличие CUDA, compute capability, память GPU и свободное место на диске.

In [ ]:
# 1. ОКРУЖЕНИЕ DATASPHERE: тот же Python и локальные артефакты, что в exp3
PROCESS_ENV = ["MPLBACKEND=Agg", "HF_HUB_OFFLINE=1", "TRANSFORMERS_OFFLINE=1",
               "HF_DATASETS_OFFLINE=1", "HF_HUB_DISABLE_TELEMETRY=1",
               "USE_TORCH=1", "USE_TF=0", "USE_FLAX=0"]
if HF_CACHE_DIR is not None:
    PROCESS_ENV += ["HF_HOME=" + str(Path(HF_CACHE_DIR).expanduser().resolve())]
assert os.path.isfile(VENV_PYTHON), "Не найден Python текущего ядра: " + VENV_PYTHON
assert sys.platform.startswith("linux"), "Этот блокнот предназначен для Linux DataSphere."
for _label, _path in (("Репозиторий", CODD_REPO_DIR), ("Веса 8B", MODEL_PATH), ("Датасет", DATA_DIR)):
    assert os.path.isdir(_path), _label + ": нет каталога " + _path
    print(_label + ": " + _path)
if ASSETS_MANIFEST:
    assert os.path.isfile(ASSETS_MANIFEST), "Нет download_manifest.json: проверьте ASSETS_DIR."
    _prepared = json.loads(Path(ASSETS_MANIFEST).read_text(encoding="utf-8"))
    assert _prepared.get("ready") is True, "Подготовка exp3-assets не завершена."
    assert (_prepared.get("model_id"), _prepared.get("hmm_id"), _prepared.get("dataset_id"),
            _prepared.get("dataset_config")) == (MODEL, HMM_CKPT, DATASET, "main"), "Перенесены другие артефакты."
    print("Ревизии перенесённых артефактов:", _prepared.get("revisions", {}))
if EXP_NAME != "exp1_llada_default":
    assert os.path.isfile(HMM_FILE), "Нет HMM: " + HMM_FILE
    print("HMM:", HMM_FILE, "| SHA-256 проверяется в preflight")
if os.path.isfile(os.path.join(DATA_DIR, "dataset_dict.json")):
    print("Датасет: DatasetDict/save_to_disk, как в exp3")
else:
    print("Датасет: локальный parquet; содержимое проверит loader")
ENV_CHECK_SNIPPET = 'import importlib, importlib.metadata as md, json, os, shutil, sys, sysconfig\nproject_dir, repo_dir, device = sys.argv[1:4]\nsys.path.insert(0, os.path.join(repo_dir, "lm-evaluation-harness"))\nsys.path.insert(0, repo_dir)\nprint("Python ядра:", sys.executable, sys.version.split()[0])\nprint("user-site доступен:", not sys.flags.no_user_site)\nfailures = []\nfor name in ("numpy", "datasets", "transformers", "pyjuice", "lm_eval.tasks",\n             "lm_eval.models.huggingface", "eval.llada.codd_llada"):\n    try:\n        module = importlib.import_module(name)\n        print("import", name, "OK", getattr(module, "__file__", ""))\n    except Exception as exc:\n        failures.append(name + ": " + type(exc).__name__ + ": " + str(exc))\n        print("import", name, "ОШИБКА:", exc)\nfor name in ("torch", "transformers", "pyjuice", "datasets", "numpy", "huggingface-hub", "accelerate", "triton"):\n    try:\n        print(name, md.version(name))\n    except md.PackageNotFoundError:\n        print(name, "нет distribution metadata")\ntry:\n    import torch\n    print("CUDA доступна:", torch.cuda.is_available(), "| сборка CUDA:", torch.version.cuda)\n    if torch.cuda.is_available():\n        torch.cuda.set_device(device)\n        p = torch.cuda.get_device_properties(torch.cuda.current_device())\n        free, total = torch.cuda.mem_get_info()\n        print("GPU:", p.name, "| bf16:", torch.cuda.is_bf16_supported())\n        print("Видеопамять: всего %.2f ГиБ, свободно %.2f ГиБ" % (total / 2**30, free / 2**30))\nexcept Exception as exc:\n    failures.append("torch/CUDA: " + str(exc))\ncc = os.environ.get("CC") or shutil.which("gcc") or shutil.which("cc") or shutil.which("clang")\ninc = sysconfig.get_paths().get("include")\nprint("Компилятор C:", cc)\nprint("Python.h:", bool(inc and os.path.isfile(os.path.join(inc, "Python.h"))))\nprint("Свободно на диске: %.1f ГиБ" % (shutil.disk_usage(project_dir).free / 2**30))\nif failures:\n    raise RuntimeError("Не хватает совместимых зависимостей текущего ядра: " + "; ".join(failures))\n'
_res = subprocess.run(
    ["env"] + PROCESS_ENV + [VENV_PYTHON] + PROCESS_PYTHON_FLAGS
    + ["-c", ENV_CHECK_SNIPPET, PROJECT_DIR, CODD_REPO_DIR, DEVICE],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True, timeout=600)
print(_res.stdout)
if _res.stderr.strip():
    print("stderr:", _res.stderr[-2500:])
assert _res.returncode == 0, "Проверка окружения не пройдена: см. сообщения выше."


## 2. Запись `exp1_core.py` и `config.json`

Ячейка записывает ядро расчёта и конфиг в `exp1_run/` и проверяет синтаксис интерпретатором venv. Её можно (и нужно) перезапускать после любой правки конфига выше: запуск процесса тоже перезаписывает `config.json`.

In [ ]:
# ===================== 2. ЗАПИСЬ exp1_core.py И config.json =====================
CORE_SOURCE = r"""# exp1_core.py - тяжёлая часть эксперимента 1 (обычная LLaDA-8B-Instruct, HMM не используется).
#
# Запускается ОТДЕЛЬНЫМ процессом из окружения venv (не из ядра блокнота):
#   env MPLBACKEND=Agg HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 HF_DATASETS_OFFLINE=1 \
#       /home/jupyter/project/venv/bin/python -I exp1_core.py --mode smoke|full|summary
# Параметры читаются из config.json рядом со скриптом (его пишет блокнот-пульт). Интернета нет.
#
# Режимы:
#   smoke   проверка окружения, загрузка модели, сверка кода, задачи, дымовой тест (копия == оригинал), пик памяти, прогноз времени
#   full    полный прогон (требует успешного smoke), кэш после каждой задачи, возобновление, RUN_LIMIT, MAX_RUNTIME_MIN
#   summary таблица точности с bootstrap-интервалами (модель не загружается)
#
# Совместимость: Python 3.10 (без вложенных одинаковых кавычек в f-строках, без match).
# НЕ ПРОВЕРЕНО: скрипт ни разу не запускался (проверен только синтаксис через ast.parse).

import argparse
import copy
import datetime
import glob
import hashlib
import importlib
import importlib.metadata as importlib_metadata
import importlib.util
import json
import math
import os
import platform
import random
import re
import shutil
import signal
import subprocess
import sys
import time
from pathlib import Path


def utc_now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")


def log(msg=""):
    stamp = datetime.datetime.now().strftime("%H:%M:%S")
    print("[" + stamp + "] " + str(msg), flush=True)


def dist_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None


def sha256_json(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")).hexdigest()


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def write_json_atomic(path, obj):
    tmp = str(path) + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, allow_nan=False)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, str(path))


# ===================== 0. КОНФИГ (из config.json) И ПРОВЕРКА ПАРАМЕТРОВ =====================
def _parse_args():
    ap = argparse.ArgumentParser(description="exp1_core: тяжёлая часть эксперимента 1 (обычная LLaDA, HMM не используется)")
    ap.add_argument("--mode", required=True, choices=["smoke", "full", "summary"])
    ap.add_argument("--config", default=str(Path(__file__).resolve().parent / "config.json"))
    return ap.parse_args()


ARGS = _parse_args()
MODE = ARGS.mode
START_UTC = utc_now()
with open(ARGS.config, encoding="utf-8") as _f:
    CFG = json.load(_f)

_REQUIRED_KEYS = [
    "MODEL", "HMM_CKPT", "DATASET", "DATASET_SPLIT", "N_TASKS", "GEN_LENGTH", "BLOCK_LENGTH", "TEMPERATURE", "NUM_STEPS_LIST",
    "REMASKING_LIST", "PC_TEMPERATURE", "PC_FRAC", "REVERSE_FRAC", "SEEDS", "EXP_NAME", "USE_PC", "SEED_USED", "SYSTEM_INSTRUCTION",
    "NUM_FEWSHOT", "TASK_NAME", "MASK_ID", "DTYPE", "ATTN_IMPLEMENTATION", "N_BOOTSTRAP", "BOOTSTRAP_SEED", "PROJECT_DIR",
    "CODD_REPO_DIR", "MODEL_PATH", "DATA_DIR", "OUT_DIR", "ALLOW_PYJUICE_STUB", "ALLOW_MODEL_CODE_MISMATCH", "STRICT_VERSIONS",
    "MIN_GPU_TOTAL_GB", "MIN_GPU_FREE_GB", "RUN_LIMIT", "MAX_RUNTIME_MIN", "REQUIRE_SMOKE",
]
_missing_keys = [k for k in _REQUIRED_KEYS if k not in CFG]
if _missing_keys:
    raise SystemExit("В config.json нет ключей: " + str(_missing_keys))

MODEL = CFG["MODEL"]
HMM_CKPT = CFG["HMM_CKPT"]
DATASET = CFG["DATASET"]
DATASET_SPLIT = CFG["DATASET_SPLIT"]
N_TASKS = CFG["N_TASKS"]
GEN_LENGTH = CFG["GEN_LENGTH"]
BLOCK_LENGTH = CFG["BLOCK_LENGTH"]
TEMPERATURE = CFG["TEMPERATURE"]
NUM_STEPS_LIST = CFG["NUM_STEPS_LIST"]
REMASKING_LIST = CFG["REMASKING_LIST"]
PC_TEMPERATURE = CFG["PC_TEMPERATURE"]
PC_FRAC = CFG["PC_FRAC"]
REVERSE_FRAC = CFG["REVERSE_FRAC"]
SEEDS = CFG["SEEDS"]
EXP_NAME = CFG["EXP_NAME"]
USE_PC = CFG["USE_PC"]
SEED_USED = CFG["SEED_USED"]
SYSTEM_INSTRUCTION = CFG["SYSTEM_INSTRUCTION"]
NUM_FEWSHOT = CFG["NUM_FEWSHOT"]
TASK_NAME = CFG["TASK_NAME"]
MASK_ID = CFG["MASK_ID"]
DTYPE = CFG["DTYPE"]
ATTN_IMPLEMENTATION = CFG["ATTN_IMPLEMENTATION"]
N_BOOTSTRAP = CFG["N_BOOTSTRAP"]
BOOTSTRAP_SEED = CFG["BOOTSTRAP_SEED"]
PROJECT_DIR = CFG["PROJECT_DIR"]
CODD_REPO_DIR = CFG["CODD_REPO_DIR"]
MODEL_PATH = CFG["MODEL_PATH"]
DATA_DIR = CFG["DATA_DIR"]
OUT_DIR = CFG["OUT_DIR"]
ALLOW_PYJUICE_STUB = CFG["ALLOW_PYJUICE_STUB"]
ALLOW_MODEL_CODE_MISMATCH = CFG["ALLOW_MODEL_CODE_MISMATCH"]
STRICT_VERSIONS = CFG["STRICT_VERSIONS"]
MIN_GPU_TOTAL_GB = CFG["MIN_GPU_TOTAL_GB"]
MIN_GPU_FREE_GB = CFG["MIN_GPU_FREE_GB"]
RUN_LIMIT = CFG["RUN_LIMIT"]
MAX_RUNTIME_MIN = CFG["MAX_RUNTIME_MIN"]
REQUIRE_SMOKE = CFG["REQUIRE_SMOKE"]

# интернета нет: принудительно офлайн; кэши HF и datasets - внутри проекта
os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["MPLBACKEND"] = "Agg"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
HF_CACHE_DIR = CFG.get("HF_CACHE_DIR") or os.path.join(PROJECT_DIR, ".hf_cache")
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["HF_DATASETS_CACHE"] = os.path.join(HF_CACHE_DIR, "datasets")

# ---------- проверки параметров (те же, что были в ячейке конфига) ----------
assert MODEL == "GSAI-ML/LLaDA-8B-Instruct", "MODEL - общий параметр, менять нельзя"
assert HMM_CKPT == "il18/llada-gsm-pc", "HMM_CKPT - общий параметр (в exp1 не используется)"
assert (DATASET, DATASET_SPLIT, N_TASKS) == ("gsm8k", "test", 300), "DATASET/DATASET_SPLIT/N_TASKS - общие параметры"
assert GEN_LENGTH == 512, "GEN_LENGTH=512 для выбранного полного прогона"
assert BLOCK_LENGTH == 32 and TEMPERATURE == 0.0, "BLOCK_LENGTH/TEMPERATURE - общие параметры"
assert NUM_STEPS_LIST == [128] and REMASKING_LIST == ["margin"], "NUM_STEPS_LIST/REMASKING_LIST - общие параметры"
assert (PC_TEMPERATURE, PC_FRAC, REVERSE_FRAC) == (0.2, 0.5, False), "PC_* - общие параметры"
assert SEEDS == [0], "SEEDS - номер повтора, согласованный с другими экспериментами"
assert SEED_USED == SEEDS[0] == 0
assert USE_PC is False, "exp1 - это эксперимент БЕЗ HMM"
assert DTYPE in ("bfloat16", "float16"), "DTYPE должен быть 'bfloat16' или 'float16', получено " + repr(DTYPE)
assert RUN_LIMIT is None or (isinstance(RUN_LIMIT, int) and 1 <= RUN_LIMIT <= N_TASKS), "RUN_LIMIT: None или целое от 1 до N_TASKS"
assert MAX_RUNTIME_MIN is None or (isinstance(MAX_RUNTIME_MIN, (int, float)) and MAX_RUNTIME_MIN > 0), "MAX_RUNTIME_MIN: None или число минут > 0"
for _gl in sorted({256, 512, GEN_LENGTH}):
    assert _gl % BLOCK_LENGTH == 0, "GEN_LENGTH=" + str(_gl) + " не делится на BLOCK_LENGTH"
    for _ns in NUM_STEPS_LIST:
        assert _ns % (_gl // BLOCK_LENGTH) == 0, "num_steps=" + str(_ns) + " не делится на число блоков при GEN_LENGTH=" + str(_gl)
NUM_BLOCKS = GEN_LENGTH // BLOCK_LENGTH

COMMON_PARAMS = {
    "MODEL": MODEL, "HMM_CKPT": HMM_CKPT, "DATASET": DATASET, "DATASET_SPLIT": DATASET_SPLIT, "N_TASKS": N_TASKS,
    "GEN_LENGTH": GEN_LENGTH, "BLOCK_LENGTH": BLOCK_LENGTH, "TEMPERATURE": TEMPERATURE,
    "NUM_STEPS_LIST": NUM_STEPS_LIST, "REMASKING_LIST": REMASKING_LIST,
    "PC_TEMPERATURE": PC_TEMPERATURE, "PC_FRAC": PC_FRAC, "REVERSE_FRAC": REVERSE_FRAC, "SEEDS": SEEDS,
}
EXP_PARAMS = dict(COMMON_PARAMS)
EXP_PARAMS.update({
    "seed_derivation": "int.from_bytes(SHA256(UTF8('exp3:{seed}:{task_id}'))[:4], 'little')",
    "EXP_NAME": EXP_NAME, "USE_PC": USE_PC, "SEED_USED": SEED_USED, "SYSTEM_INSTRUCTION": SYSTEM_INSTRUCTION,
    "NUM_FEWSHOT": NUM_FEWSHOT, "TASK_NAME": TASK_NAME, "MASK_ID": MASK_ID,
    "DTYPE": DTYPE, "ATTN_IMPLEMENTATION": ATTN_IMPLEMENTATION,
})
COMMON_SHA256 = sha256_json(COMMON_PARAMS)
assert COMMON_SHA256 == "d9fc8dc825bdeecb34b0c190d445c02860b6a75b45a4a52c95882a3effb9f4ef", "Хэш выбранной сетки не совпал"
EXP_PARAMS["runtime_profile"] = {
    "name": "exp3-assets-kernel-python-v1", "repo_dir": CODD_REPO_DIR,
    "model_path": MODEL_PATH, "data_dir": DATA_DIR, "device": CFG["DEVICE"], "offline": True,
    "hmm_path": CFG.get("HMM_LOCAL_PATH") if EXP_NAME != "exp1_llada_default" else None,
}
EXP_SHA256 = sha256_json(EXP_PARAMS)

ENV_PATH = os.path.join(OUT_DIR, "env.json")
CONFIG_PATH = os.path.join(OUT_DIR, "config.json")
TASKS_PATH = os.path.join(OUT_DIR, "tasks.json")
SMOKE_PATH = os.path.join(OUT_DIR, "smoke.json")
SUMMARY_PATH = os.path.join(OUT_DIR, "summary.json")
STATUS_PATH = os.path.join(OUT_DIR, "status.json")

try:
    import numpy as np
except Exception as _e:
    raise SystemExit("numpy не импортируется в этом окружении: " + str(_e))

# глобальное состояние, заполняется по ходу работы
torch = None
F = None
TORCH_DTYPE = None
ENV_INFO = {}
PYJUICE_MODE = None
model = None
tokenizer = None
PLAIN = None
lm = None
task = None
INSTANCES = []
TASK_IDS = []
PROMPT_IDS = []
UNTIL = []
IDS_SHA256 = None
QUESTIONS_SHA256 = None
MODEL_CODE_CHECK = {}
STOP_REQUESTED = {"flag": False}


def set_status(state, message=""):
    os.makedirs(OUT_DIR, exist_ok=True)
    write_json_atomic(STATUS_PATH, {"mode": MODE, "state": state, "message": message, "pid": os.getpid(), "updated_utc": utc_now()})
    log("[EXP1] STATUS: " + state + " " + message)


def _on_stop_signal(signum, frame):
    if STOP_REQUESTED["flag"]:
        log("повторный сигнал остановки: немедленный выход (кэш к этому моменту согласован)")
        os._exit(143)
    STOP_REQUESTED["flag"] = True
    log("получен сигнал остановки: заканчиваю текущую задачу, дописываю кэш и steplog, затем остановлюсь")


# ===================== 1. ПРОВЕРКА ОКРУЖЕНИЯ И GPU =====================
EXPECTED_VERSIONS = {"torch": "2.6.0", "transformers": "4.57.3", "pyjuice": "2.4.1", "huggingface-hub": "0.36.0", "numpy": "2.2.6"}
REQUIRED_IMPORTS = ["accelerate", "datasets", "evaluate", "jinja2", "jsonlines", "pytablewriter", "rouge_score", "sacrebleu", "sklearn",
                    "sqlitedict", "zstandard", "dill", "word2number", "more_itertools", "pyarrow", "safetensors", "tokenizers"]
OPTIONAL_IMPORTS = ["peft"]     # не нужен для расчёта: отсутствие только предупреждение
REQUIRED_REPO_FILES = [
    "eval/llada/codd_llada.py", "eval/llada/llada_generate.py", "eval/llada/modeling_llada.py",
    "eval/llada/configuration_llada.py", "codd/model_class.py", "codd/__init__.py",
    "eval/eval.py", "eval/harness.py", "lm-evaluation-harness/lm_eval/__init__.py",
    "lm-evaluation-harness/lm_eval/tasks/gsm8k/gsm8k.yaml", "lm-evaluation-harness/pyproject.toml",
]


def check_local_assets():
    # Проверяем только локальные файлы; сеть, установка и скачивание отключены.
    assert CFG.get("OFFLINE") is True, "Требуется OFFLINE=True."
    assert os.path.isdir(CODD_REPO_DIR), "Нет репозитория: " + CODD_REPO_DIR
    assert os.path.isdir(PROJECT_DIR), "Нет постоянного хранилища: " + PROJECT_DIR
    assert os.path.isdir(MODEL_PATH), "Нет распакованного snapshot: " + MODEL_PATH
    assert os.path.isdir(DATA_DIR), "Нет локального датасета: " + DATA_DIR
    for filename in ("pc_model.jpc", "codd_config.json"):
        assert not os.path.exists(os.path.join(MODEL_PATH, filename)), (
            "Нужен snapshot базовой LLaDA, без встроенной PC: " + filename)
    manifest = CFG.get("ASSETS_MANIFEST")
    if not manifest:
        return {"status": "НЕ ПРОВЕРЕНО: ручные пути, ревизии артефактов неизвестны"}
    assert os.path.isfile(manifest), "Нет manifest: " + manifest
    with open(manifest, encoding="utf-8") as stream:
        prepared = json.load(stream)
    assert prepared.get("ready") is True, "Подготовка exp3-assets не завершена"
    assert (prepared.get("model_id"), prepared.get("hmm_id"), prepared.get("dataset_id"),
            prepared.get("dataset_config")) == (MODEL, HMM_CKPT, DATASET, "main"), "Артефакты другого эксперимента"
    return {"status": "MATCH", "manifest": manifest, "revisions": prepared.get("revisions", {})}


def check_environment():
    global torch, F, TORCH_DTYPE, ENV_INFO
    log("Python: " + sys.version.split()[0] + " | " + sys.executable + " | " + platform.platform())
    assert sys.version_info >= (3, 10), "Нужен Python >= 3.10"
    try:
        import torch as _torch
        import torch.nn.functional as _F
    except Exception as e:
        raise RuntimeError("torch не импортируется в Python текущего ядра: " + str(e)) from e
    torch = _torch
    F = _F
    TORCH_DTYPE = {"bfloat16": torch.bfloat16, "float16": torch.float16}[DTYPE]
    prepared_assets = check_local_assets()
    torch.set_grad_enabled(False)
    if torch.cuda.is_available():
        torch.cuda.set_device(CFG["DEVICE"])
    log("torch: " + torch.__version__ + " | CUDA (сборка torch): " + str(torch.version.cuda) + " | DTYPE: " + DTYPE)
    assert torch.cuda.is_available(), "GPU не найден (torch.cuda.is_available() == False): нужен запуск на GPU"
    gpu_index = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(gpu_index)
    free, total = torch.cuda.mem_get_info()
    gib = 1024 ** 3
    log("GPU: " + props.name + " | compute capability " + str(props.major) + "." + str(props.minor))
    log("Видеопамять фактически: всего " + format(total / gib, ".2f") + " ГиБ (" + str(total) + " байт), свободно " + format(free / gib, ".2f") + " ГиБ (" + str(free) + " байт)")
    assert total / gib >= MIN_GPU_TOTAL_GB, "Памяти GPU " + format(total / gib, ".2f") + " ГиБ < MIN_GPU_TOTAL_GB=" + str(MIN_GPU_TOTAL_GB)
    assert free / gib >= MIN_GPU_FREE_GB, "Свободно " + format(free / gib, ".2f") + " ГиБ < MIN_GPU_FREE_GB=" + str(MIN_GPU_FREE_GB) + ": освободите GPU (другие процессы)"
    if DTYPE == "bfloat16":
        assert props.major >= 8 and torch.cuda.is_bf16_supported(), "Нужна аппаратная поддержка bfloat16, как в exp3"
    disk = shutil.disk_usage(PROJECT_DIR)
    log("Диск (" + PROJECT_DIR + "): свободно " + format(disk.free / gib, ".1f") + " ГиБ из " + format(disk.total / gib, ".1f"))
    os.makedirs(OUT_DIR, exist_ok=True)
    probe = os.path.join(OUT_DIR, ".write_probe")
    with open(probe, "w", encoding="utf-8") as f:
        f.write("ok")
    os.remove(probe)

    missing_files = [p for p in REQUIRED_REPO_FILES if not os.path.isfile(os.path.join(CODD_REPO_DIR, p))]
    assert not missing_files, "В CODD_REPO_DIR=" + CODD_REPO_DIR + " нет файлов: " + str(missing_files)
    weights_dir = Path(MODEL_PATH)
    assert weights_dir.is_dir(), "Нет папки весов: " + MODEL_PATH
    names = set(os.listdir(MODEL_PATH))
    for need in ("config.json", "tokenizer.json", "modeling_llada.py", "configuration_llada.py", "model.safetensors.index.json"):
        assert need in names, "В " + MODEL_PATH + " нет файла " + need + " (склеены ли шарды и скопированы ли файлы?)"
    shards = sorted(n for n in names if n.endswith(".safetensors"))
    assert len(shards) == 6, "Ожидалось 6 шардов *.safetensors (склеенных), найдено " + str(len(shards)) + ": " + str(shards)
    assert not any(".part" in n for n in names), "В папке весов остались не склеенные части *.partNNN"
    weights_bytes = sum(os.path.getsize(os.path.join(MODEL_PATH, n)) for n in shards)
    log("Веса: " + str(len(shards)) + " шардов, " + format(weights_bytes / 1e9, ".2f") + " ГБ")
    assert weights_bytes > 15_000_000_000, "суммарный размер шардов " + str(weights_bytes) + " байт подозрительно мал (ожидается около 16.0 ГБ): склейка неполная?"

    missing_imports = [m for m in REQUIRED_IMPORTS if importlib.util.find_spec(m) is None]
    assert not missing_imports, "В Python текущего ядра нет пакетов: " + str(missing_imports) + " (автоустановка отключена; используйте подготовленную папку python-deps)"
    missing_optional = [m for m in OPTIONAL_IMPORTS if importlib.util.find_spec(m) is None]
    if missing_optional:
        log("ПРЕДУПРЕЖДЕНИЕ: нет необязательных пакетов " + str(missing_optional) + " (на расчёт не влияет)")

    log("Версии (ориентир - сохранённое окружение exp3; фактические версии записываются):")
    mismatches = []
    for name, want in EXPECTED_VERSIONS.items():
        have = dist_version(name)
        ok = have is not None and have.split("+")[0] == want
        if not ok:
            mismatches.append([name, want, have])
        log("   " + name.ljust(18) + (want or "").ljust(10) + str(have).ljust(16) + ("OK" if ok else "РАСХОЖДЕНИЕ"))
    if mismatches:
        msg = "Расхождения версий: " + str(mismatches)
        if STRICT_VERSIONS:
            raise RuntimeError(msg)
        log("ПРЕДУПРЕЖДЕНИЕ (не ошибка): " + msg)
    log("НЕ ПРОВЕРЕНО: совместимость этих версий с кодом модели (modeling_llada.py), CoDD и lm_eval; проверена только сборка бандла.")
    ENV_INFO = {
        "started_utc": START_UTC, "mode": MODE, "python": sys.version.split()[0], "python_executable": sys.executable,
        "platform": platform.platform(), "torch": torch.__version__, "torch_cuda_build": torch.version.cuda,
        "gpu_name": props.name, "gpu_total_gb": round(total / gib, 2), "gpu_total_bytes": total,
        "gpu_capability": str(props.major) + "." + str(props.minor), "dtype": DTYPE,
        "hf_reachable": "offline (HF_HUB_OFFLINE=1)", "pypi_reachable": "offline",
        "project_dir": PROJECT_DIR, "model_path": MODEL_PATH, "local_files_only": True, "hf_cache_dir": HF_CACHE_DIR,
        "transformers": dist_version("transformers"), "huggingface_hub": dist_version("huggingface-hub"), "numpy": dist_version("numpy"),
        "version_mismatches_vs_expected": mismatches, "weights_gib": round(weights_bytes / gib, 2),
    }
    ENV_INFO.update({"artifact_preparation": prepared_assets, "device": CFG["DEVICE"],
                     "runtime_profile": "exp3-assets-kernel-python-v1", "data_dir": DATA_DIR,
                     "python_user_site_enabled": not sys.flags.no_user_site})
    return gpu_index, total


# ===================== 2. PYJUICE И КОД РЕПОЗИТОРИЯ =====================
def prepare_pyjuice():
    global PYJUICE_MODE
    try:
        import pyjuice  # noqa: F401
        PYJUICE_MODE = "real"
        log("pyjuice импортирован, версия " + str(dist_version("pyjuice")) + " (ожидается 2.4.1)")
    except Exception as e:
        for k in [k for k in sys.modules if k == "pyjuice" or k.startswith("pyjuice.")]:
            del sys.modules[k]
        log("pyjuice НЕ импортируется: " + type(e).__name__ + ": " + str(e))
        if not ALLOW_PYJUICE_STUB:
            raise RuntimeError("pyjuice недоступен. Для exp1 (HMM не используется) можно поставить ALLOW_PYJUICE_STUB=true в config.json, но ТОЛЬКО в exp1: в блокнотах 2-5 заглушка запрещена.") from e
        import types
        stub = types.ModuleType("pyjuice")

        def _forbidden(*args, **kwargs):
            raise RuntimeError("pyjuice - ЗАГЛУШКА (exp1). Путь HMM здесь вызываться не должен.")

        stub.load = _forbidden
        stub.compile = _forbidden
        stub.save = _forbidden
        stub.queries = types.SimpleNamespace(sample=_forbidden)
        stub.__is_stub__ = True
        sys.modules["pyjuice"] = stub
        PYJUICE_MODE = "stub"
        log("=" * 90)
        log("ВНИМАНИЕ: ИСПОЛЬЗУЕТСЯ ЗАГЛУШКА pyjuice. Допустимо ТОЛЬКО в exp1 (use_pc=False). В блокнотах 2-5 заглушка ЗАПРЕЩЕНА.")
        log("=" * 90)
    ENV_INFO["pyjuice"] = dist_version("pyjuice") if PYJUICE_MODE == "real" else "stub"
    ENV_INFO["pyjuice_mode"] = PYJUICE_MODE


def git_info(repo):
    def run(args):
        return subprocess.run(["git", "-C", repo] + args, capture_output=True, text=True, timeout=20)
    try:
        head = run(["rev-parse", "HEAD"])
        if head.returncode != 0:
            return {"commit": "unknown (нет git-метаданных)", "dirty": None}
        status = run(["status", "--porcelain"])
        return {"commit": head.stdout.strip(), "dirty": bool(status.stdout.strip())}
    except Exception as e:
        return {"commit": "unknown (" + type(e).__name__ + ")", "dirty": None}


def import_repo_code():
    global CoddLlada, LLaDAModelLM, ORIG_GEN_MODULE, llada_diffusion_generate, add_gumbel_noise, get_num_transfer_tokens, _is_codd_model
    global lm_eval, TaskManager, get_task_dict, HFLM, handle_stop_sequences, postprocess_generated_text
    for p in (os.path.join(CODD_REPO_DIR, "lm-evaluation-harness"), CODD_REPO_DIR):
        if p in sys.path:
            sys.path.remove(p)
        sys.path.insert(0, p)    # репозиторий перекрывает любые другие версии (в -I режиме cwd не в sys.path)
    importlib.invalidate_caches()
    from eval.llada.codd_llada import CoddLlada
    from eval.llada.modeling_llada import LLaDAModelLM
    from eval.llada import llada_generate as ORIG_GEN_MODULE
    from eval.llada.llada_generate import llada_diffusion_generate, add_gumbel_noise, get_num_transfer_tokens, _is_codd_model
    import lm_eval
    from lm_eval.tasks import TaskManager, get_task_dict
    from lm_eval.models.huggingface import HFLM
    from lm_eval.models.utils import handle_stop_sequences, postprocess_generated_text
    repo_real = os.path.realpath(CODD_REPO_DIR) + os.sep
    for name, mod in (("eval.llada.llada_generate", ORIG_GEN_MODULE), ("lm_eval", lm_eval)):
        f = os.path.realpath(mod.__file__)
        assert f.startswith(repo_real), name + " импортирован НЕ из репозитория: " + f + " (ожидалась папка " + repo_real + ")"
    log("Код репозитория импортирован из: " + CODD_REPO_DIR)
    pyproject = open(os.path.join(CODD_REPO_DIR, "lm-evaluation-harness", "pyproject.toml"), encoding="utf-8").read()
    m = re.search(r'^version\s*=\s*"([^"]+)"', pyproject, re.M)
    git = git_info(CODD_REPO_DIR)
    ENV_INFO.update({
        "lm_eval_version_pyproject": m.group(1) if m else None, "codd_commit": git["commit"], "codd_dirty": git["dirty"],
        "llada_generate_py_sha256": sha256_file(os.path.join(CODD_REPO_DIR, "eval", "llada", "llada_generate.py")),
        "common_params_sha256": COMMON_SHA256, "exp_params_sha256": EXP_SHA256,
    })


def write_env_and_config():
    if os.path.exists(ENV_PATH):
        with open(ENV_PATH, encoding="utf-8") as f:
            old_env = json.load(f)
        prev = old_env.pop("previous_sessions", [])
        ENV_INFO["previous_sessions"] = prev + [old_env]     # история прежних сессий (smoke/full/возобновление)
    write_json_atomic(ENV_PATH, ENV_INFO)
    log("env.json записан: " + ENV_PATH)
    config_doc = {"common_params": COMMON_PARAMS, "common_sha256": COMMON_SHA256, "exp_params": EXP_PARAMS, "exp_sha256": EXP_SHA256}
    if os.path.exists(CONFIG_PATH):
        with open(CONFIG_PATH, encoding="utf-8") as f:
            old_cfg = json.load(f)
        if old_cfg.get("exp_sha256") != EXP_SHA256:
            keys = set(old_cfg.get("exp_params", {})) | set(EXP_PARAMS)
            diff = {}
            for k in keys:
                a, b = old_cfg.get("exp_params", {}).get(k), EXP_PARAMS.get(k)
                if a != b:
                    diff[k] = [a, b]
            raise RuntimeError("В " + OUT_DIR + " уже лежат результаты с ДРУГИМ конфигом (параметр: [было, стало]): " + str(diff) + ". Верните прежние значения или укажите другой OUT_DIR.")
        log("config.json совпадает с текущим конфигом - возобновление допустимо.")
    else:
        write_json_atomic(CONFIG_PATH, config_doc)
        log("config.json записан: " + CONFIG_PATH)
    log("codd_commit: " + str(ENV_INFO.get("codd_commit")) + " | dirty: " + str(ENV_INFO.get("codd_dirty")))


# ===================== 3. ЗАГРУЗКА МОДЕЛИ И СВЕРКА КОДА =====================
class PlainView:
    # Та же модель как "обычная": вызов базового forward без CoDD-аргументов. Атрибута pc_model нет специально:
    # _is_codd_model(PlainView(...)) == False, поэтому оригинальная функция идёт по базовой ветке (model(x).logits).
    def __init__(self, m):
        self._m = m

    @property
    def device(self):
        return self._m.device

    def __call__(self, x):
        return LLaDAModelLM.forward(self._m, x)


def load_model():
    global model, tokenizer, PLAIN
    t0 = time.time()
    log("Загрузка модели с диска (local_files_only=True) из " + MODEL_PATH)
    class LocalCoddLlada(CoddLlada):
        @staticmethod
        def _resolve_file(path_or_repo, filename):
            # Как в WordsOnlyLlada из exp3: отсутствующий файл не ищется на Hub.
            candidate = os.path.join(path_or_repo, filename)
            return candidate if os.path.isfile(candidate) else None
    model = LocalCoddLlada.from_pretrained(
        MODEL_PATH,
        torch_dtype=TORCH_DTYPE,
        local_files_only=True,
        attn_implementation=ATTN_IMPLEMENTATION,
        device_map=CFG["DEVICE"],
    )
    ENV_INFO["model_load_seconds"] = round(time.time() - t0, 1)
    log("Модель загружена за " + format(time.time() - t0, ".0f") + " c")
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    tokenizer = model.tokenizer
    assert tokenizer is not None, "model.tokenizer is None: токенизатор не загрузился"
    assert model.pc_model is None, "pc_model должен быть None: в exp1 HMM не используется и не должен загружаться"
    assert model.codd_config.mask_id == MASK_ID, "mask_id модели " + str(model.codd_config.mask_id) + " != " + str(MASK_ID)
    emb = model.get_input_embeddings().weight
    assert emb.shape[0] > MASK_ID, "Размер словаря " + str(emb.shape[0]) + " не содержит mask_id=" + str(MASK_ID)
    assert next(model.parameters()).dtype == TORCH_DTYPE, "ожидается " + DTYPE + ", а параметры модели в " + str(next(model.parameters()).dtype)
    assert next(model.parameters()).device.type == "cuda", "модель должна быть на GPU"
    assert not model.training, "модель должна быть в eval-режиме"
    log("dtype модели: " + str(next(model.parameters()).dtype) + " | mask_id " + str(MASK_ID) + " -> токен " + str(tokenizer.convert_ids_to_tokens(MASK_ID)) + " | eos_token_id " + str(tokenizer.eos_token_id))
    log("Память GPU занята: " + format(torch.cuda.memory_allocated() / 1024 ** 3, ".1f") + " ГиБ")
    PLAIN = PlainView(model)
    assert _is_codd_model(model) is True and _is_codd_model(PLAIN) is False


def check_model_code():
    # вместо загрузки второй модели: sha256 исходников из репозитория против файлов рядом с весами
    global MODEL_CODE_CHECK
    def sha_norm(path):
        with open(path, "rb") as f:
            # CRLF/LF и завершающие LF не меняют Python-код; raw SHA-256 сохраняются ниже.
            return hashlib.sha256(f.read().replace(b"\r\n", b"\n").rstrip(b"\n")).hexdigest()
    for fname in ("modeling_llada.py", "configuration_llada.py"):
        local = os.path.join(CODD_REPO_DIR, "eval", "llada", fname)
        snap = os.path.join(MODEL_PATH, fname)
        if not os.path.isfile(snap):
            MODEL_CODE_CHECK[fname] = {"verdict": "НЕ ПРОВЕРЕНО", "reason": "нет файла " + fname + " рядом с весами"}
            log(fname + ": НЕ ПРОВЕРЕНО - нет файла рядом с весами")
            continue
        a, b = sha256_file(local), sha256_file(snap)
        MODEL_CODE_CHECK[fname] = {"verdict": "MATCH" if a == b else "MISMATCH", "repo_sha256": a, "snapshot_sha256": b,
                                   "snapshot_source": "локальная папка весов", "equal_after_newline_normalization": sha_norm(local) == sha_norm(snap)}
        MODEL_CODE_CHECK[fname]["comparison_normalization"] = "CRLF -> LF; ignore trailing LF bytes"
        if a != b and MODEL_CODE_CHECK[fname]["equal_after_newline_normalization"]:
            MODEL_CODE_CHECK[fname]["verdict"] = "MATCH_NORMALIZED_NEWLINES"
            log(fname + ": различаются только переводы строк/завершающий LF; содержимое кода совпадает")
        log(fname + ": " + MODEL_CODE_CHECK[fname]["verdict"] + " repo=" + a[:16] + "... snapshot=" + b[:16] + "...")
    ENV_INFO["model_code_check"] = MODEL_CODE_CHECK
    write_json_atomic(ENV_PATH, ENV_INFO)
    bad = [k for k, v in MODEL_CODE_CHECK.items() if v["verdict"] == "MISMATCH"]
    if bad and not ALLOW_MODEL_CODE_MISMATCH:
        raise RuntimeError("Исходники модели из репозитория CoDD отличаются от файлов рядом с весами: " + str(bad) + ". Остановка до решения автора (ALLOW_MODEL_CODE_MISMATCH=true, чтобы продолжить осознанно).")


# ===================== 4. ЗАДАЧИ GSM8K (ОФЛАЙН-parquet ЧЕРЕЗ КОД lm_eval) =====================
def find_parquet(split):
    patterns = [os.path.join("main", split + "-*.parquet"), split + "-*.parquet", os.path.join("main", split + ".parquet"), split + ".parquet"]
    for pat in patterns:
        hits = sorted(glob.glob(os.path.join(DATA_DIR, pat)))
        if len(hits) == 1:
            return hits[0]
        if len(hits) > 1:
            raise RuntimeError("В " + DATA_DIR + " несколько файлов для сплита " + split + ": " + str(hits))
    return None


def load_task():
    # Источник данных как в exp3; исходные шаблоны, фильтры и метрика GSM8K не изменяются.
    if os.path.isfile(os.path.join(DATA_DIR, "dataset_dict.json")):
        from datasets import DatasetDict, load_from_disk
        loaded = load_from_disk(DATA_DIR)
        assert isinstance(loaded, DatasetDict) and {"train", "test"} <= set(loaded), (
            "Нужен DatasetDict с train/test, как в exp3: " + DATA_DIR)
        assert all({"question", "answer"} <= set(loaded[s].column_names) for s in ("train", "test"))
        assert len(loaded["test"]) >= N_TASKS, "Недостаточно test-задач"
        override = {"task": TASK_NAME, "custom_dataset": lambda **kwargs: loaded, "dataset_kwargs": {}}
    else:
        train, test = find_parquet("train"), find_parquet("test")
        assert train is not None and test is not None, "Нужны train/test parquet либо DatasetDict/save_to_disk: " + DATA_DIR
        override = {"task": TASK_NAME, "dataset_path": "parquet", "dataset_name": None,
                    "dataset_kwargs": {"data_files": {"train": train, "test": test}}}
    task_dict = get_task_dict([override], task_manager=TaskManager())
    assert len(task_dict) == 1, "Ожидалась одна задача " + TASK_NAME
    loaded_task = next(iter(task_dict.values()))
    loaded_task.set_fewshot_seed(0)
    return loaded_task


def gold_final(answer_text):
    m = re.search(r"####\s*([^\n]+)", answer_text)
    return m.group(1).strip() if m else None


def setup_tasks():
    global task, lm, INSTANCES, TASK_IDS, PROMPT_IDS, UNTIL, IDS_SHA256, QUESTIONS_SHA256, MODEL_MAX_LEN
    task = load_task()
    task.set_config(key="num_fewshot", value=NUM_FEWSHOT)
    assert task.get_config("num_fewshot") == NUM_FEWSHOT == 0
    assert task.OUTPUT_TYPE == "generate_until", "ожидался generate_until, а не " + str(task.OUTPUT_TYPE)
    gk = task.get_config("generation_kwargs")
    assert gk.get("do_sample") is False and float(gk.get("temperature", 0.0)) == TEMPERATURE == 0.0, "generation_kwargs задачи не жадные: " + str(gk)
    n_docs = len(task.eval_docs)
    log("Документов в " + DATASET_SPLIT + "-выборке: " + str(n_docs) + ("" if n_docs == 1319 else " (ВНИМАНИЕ: у стандартного GSM8K test 1319 задач)"))
    assert n_docs >= N_TASKS, "в выборке " + str(n_docs) + " < N_TASKS=" + str(N_TASKS)
    assert {f.name for f in task._filters} == {"strict-match", "flexible-extract"}, "неожиданные фильтры задачи: " + str([f.name for f in task._filters])

    lm = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=1)
    log("HFLM backend: " + str(lm.backend) + " | max_length (HFLM): " + str(lm.max_length) + " | tokenizer_name: " + str(lm.tokenizer_name))
    task.build_all_requests(
        limit=N_TASKS, rank=0, world_size=1, cache_requests=False, rewrite_requests_cache=False,
        system_instruction=SYSTEM_INSTRUCTION, apply_chat_template=True, fewshot_as_multiturn=False,
        chat_template=lm.apply_chat_template, tokenizer_name=lm.tokenizer_name,
    )
    INSTANCES = list(task.instances)
    assert len(INSTANCES) == N_TASKS, "получено " + str(len(INSTANCES)) + " запросов вместо " + str(N_TASKS)
    for i, inst in enumerate(INSTANCES):
        assert inst.doc_id == i, "порядок задач нарушен: позиция " + str(i) + ", doc_id " + str(inst.doc_id)
        assert inst.request_type == "generate_until" and inst.repeats == 1
    TASK_IDS = [inst.doc_id for inst in INSTANCES]
    assert TASK_IDS == list(range(N_TASKS))

    gen_kwargs = copy.deepcopy(INSTANCES[0].args[1])
    UNTIL = handle_stop_sequences(gen_kwargs.pop("until", None), eos=lm.tok_decode(lm.eot_token_id, skip_special_tokens=False))
    log("until (стоп-последовательности из задачи + EOS): " + str(UNTIL))
    assert all(inst.args[1] == INSTANCES[0].args[1] for inst in INSTANCES), "generation_kwargs различаются между задачами"

    PROMPT_IDS = []
    for inst in INSTANCES:
        ids, _ = lm.tok_batch_encode([inst.args[0]], left_truncate_len=None, truncation=False)
        assert ids.shape[0] == 1
        PROMPT_IDS.append(ids[0].tolist())
        assert MASK_ID not in PROMPT_IDS[-1], "в промпте встретился mask_id"
    limits = [lm.max_length]
    if hasattr(model.config, "max_sequence_length") and model.config.max_sequence_length:
        limits.append(int(model.config.max_sequence_length))
    MODEL_MAX_LEN = min(limits)
    max_prompt = max(len(p) for p in PROMPT_IDS)
    for gl in sorted({256, 512, GEN_LENGTH}):
        assert max_prompt + gl <= MODEL_MAX_LEN, "Самый длинный промпт (" + str(max_prompt) + " ток.) + GEN_LENGTH=" + str(gl) + " > лимита длины модели " + str(MODEL_MAX_LEN)
    log("Промпты: длина " + str(min(len(p) for p in PROMPT_IDS)) + ".." + str(max_prompt) + " токенов; лимит модели " + str(MODEL_MAX_LEN) + "; GEN_LENGTH=" + str(GEN_LENGTH) + " помещается (проверено и для 256, и для 512).")
    log("-" * 80)
    log("ПРОМПТ задачи 0 (как видит модель):")
    print(INSTANCES[0].args[0], flush=True)
    log("-" * 80)

    IDS_SHA256 = hashlib.sha256(json.dumps(TASK_IDS, separators=(",", ":")).encode("utf-8")).hexdigest()
    QUESTIONS_SHA256 = hashlib.sha256(json.dumps([inst.doc["question"] for inst in INSTANCES], ensure_ascii=False, separators=(",", ":")).encode("utf-8")).hexdigest()
    tasks_doc = {
        "dataset": DATASET, "split": DATASET_SPLIT, "n_tasks": N_TASKS, "data_source": "локальные parquet через lm_eval (dataset_path=parquet): " + DATA_DIR,
        "system_instruction": SYSTEM_INSTRUCTION, "num_fewshot": NUM_FEWSHOT, "chat_template_applied": True,
        "ids_sha256": IDS_SHA256,
        "ids_sha256_definition": 'sha256(json.dumps(ids, separators=(",",":")).encode("utf-8")), ids = [0..N_TASKS-1]',
        "questions_sha256": QUESTIONS_SHA256, "ids": TASK_IDS,
        "tasks": [
            {"task_id": inst.doc_id, "question": inst.doc["question"], "gold_final": gold_final(inst.doc["answer"]),
             "prompt_sha256": hashlib.sha256(inst.args[0].encode("utf-8")).hexdigest()}
            for inst in INSTANCES
        ],
    }
    assert all(t["gold_final"] is not None for t in tasks_doc["tasks"]), "у некоторых задач нет '#### ответ' в эталоне"
    if os.path.exists(TASKS_PATH):
        with open(TASKS_PATH, encoding="utf-8") as f:
            old = json.load(f)
        assert old["ids_sha256"] == IDS_SHA256, "список id отличается от сохранённого в tasks.json - возобновление небезопасно"
        assert old["questions_sha256"] == QUESTIONS_SHA256, "тексты вопросов отличаются от сохранённых в tasks.json (другая версия датасета?)"
        old_prompts = {t["task_id"]: t["prompt_sha256"] for t in old["tasks"]}
        new_prompts = {t["task_id"]: t["prompt_sha256"] for t in tasks_doc["tasks"]}
        assert old_prompts == new_prompts, "промпты отличаются от сохранённых в tasks.json (другой шаблон чата/токенизатор?) - возобновление небезопасно"
        log("tasks.json совпадает с текущими задачами и промптами - возобновление допустимо.")
    else:
        write_json_atomic(TASKS_PATH, tasks_doc)
        log("tasks.json записан: " + TASKS_PATH)
    log("=" * 80)
    log("sha256 списка id задач: " + IDS_SHA256)
    log("(справочно) sha256 текстов вопросов: " + QUESTIONS_SHA256)
    log("=" * 80)


# ===================== 5. КОПИЯ llada_diffusion_generate С ЖУРНАЛОМ ШАГОВ =====================
def llada_generate_logged(model, prompt, num_steps=128, gen_length=128, block_length=128,
                          temperature=0., cfg_scale=0., remasking='low_confidence', mask_id=126336,
                          use_pc=False, pc_temperature=None, pc_frac=None, reverse_frac=None):
    # Копия llada_diffusion_generate (CoDD/eval/llada/llada_generate.py, коммит d9760f4) + журнал шагов.
    # Отличия от оригинала (на вычисления не влияют): assert use_pc is False; нет tqdm; steplog; topk сохраняет значения.
    assert use_pc is False, "exp1: HMM не используется (use_pc=False)"
    assert prompt.dim() == 2 and prompt.shape[0] == 1, "ожидается промпт формы (1, L)"
    with torch.no_grad():
        is_codd = _is_codd_model(model)

        x = torch.full((1, prompt.shape[1] + gen_length), mask_id, dtype=torch.long).to(model.device)
        x[:, :prompt.shape[1]] = prompt.clone()

        prompt_index = (x != mask_id)

        assert gen_length % block_length == 0
        num_blocks = gen_length // block_length

        assert num_steps % num_blocks == 0
        steps = num_steps // num_blocks

        steplog = []
        for num_block in range(num_blocks):
            start_idx = prompt.shape[1] + num_block * block_length
            end_idx = prompt.shape[1] + (num_block + 1) * block_length

            block_mask_index = (x[:, start_idx:end_idx] == mask_id)
            num_transfer_tokens = get_num_transfer_tokens(block_mask_index, steps)

            for i in range(steps):
                mask_index = (x == mask_id)
                n_masked_block_before = int(mask_index[0, start_idx:end_idx].sum().item())

                # ---------- forward pass ----------
                if is_codd:
                    # CoDD.forward() handles CFG + PC modification internally
                    pc_block_range = (start_idx, end_idx) if use_pc else None
                    output = model(
                        x,
                        pc_block_range=pc_block_range,
                        use_pc=use_pc,
                        cfg_scale=cfg_scale,
                        prompt_index=prompt_index if cfg_scale > 0. else None,
                        pc_temperature=pc_temperature,
                        pc_frac=pc_frac,
                        reverse_frac=reverse_frac,
                    )
                    logits = output.logits
                else:
                    # Plain base model path (backward compatible)
                    if cfg_scale > 0.:
                        un_x = x.clone()
                        un_x[prompt_index] = mask_id
                        x_ = torch.cat([x, un_x], dim=0)
                        logits = model(x_).logits
                        logits, un_logits = torch.chunk(logits, 2, dim=0)
                        logits = un_logits + (cfg_scale + 1) * (logits - un_logits)
                    else:
                        logits = model(x).logits

                # ---------- sampling ----------
                logits_with_noise = add_gumbel_noise(logits, temperature=temperature)
                x0 = torch.argmax(logits_with_noise, dim=-1)

                if remasking == 'low_confidence':
                    p = F.softmax(logits.to(torch.float64), dim=-1)
                    x0_p = torch.squeeze(
                        torch.gather(p, dim=-1, index=torch.unsqueeze(x0, -1)), -1)
                elif remasking == 'random':
                    x0_p = torch.rand((x0.shape[0], x0.shape[1]), device=x0.device)
                elif remasking == 'margin':
                    probs = torch.softmax(logits_with_noise, dim=-1)
                    top_probs, _ = torch.topk(probs, k=2, dim=-1)
                    x0_p = top_probs[..., 0] - top_probs[..., 1]
                elif remasking == 'entropy':
                    probs = torch.softmax(logits_with_noise, dim=-1)
                    epsilon = 1e-10
                    log_probs = torch.log(probs + epsilon)
                    x0_p = torch.sum(probs * log_probs, dim=-1)
                else:
                    raise NotImplementedError(remasking)

                x0_p[:, end_idx:] = -np.inf

                x0 = torch.where(mask_index, x0, x)
                confidence = torch.where(mask_index, x0_p, -np.inf)

                for j in range(confidence.shape[0]):
                    num_tokens = num_transfer_tokens[j, i].item()
                    if num_tokens > 0:
                        conf_vals, select_index = torch.topk(confidence[j], k=num_tokens)
                        x[j, select_index] = x0[j, select_index]
                        pos_abs = select_index.tolist()
                        steplog.append({
                            "block": num_block,
                            "step_in_block": i,
                            "global_step": num_block * steps + i,
                            "n_masked_block_before": n_masked_block_before,
                            "positions_abs": pos_abs,
                            "positions_rel": [p_ - prompt.shape[1] for p_ in pos_abs],
                            "token_ids": x0[j, select_index].tolist(),
                            "confidence": [float(c) for c in conf_vals.float().tolist()],
                        })

    return x, steplog


# ===================== 6. ДЕКОДИРОВАНИЕ, ПРОВЕРКА ОТВЕТА (КОД lm_eval), КЭШ =====================
FILTER_NAMES = ("strict-match", "flexible-extract")
REQUIRED_ROW_KEYS = ("task_id", "seed", "correct", "prediction", "num_steps", "remasking", "pc_active_steps", "delta_words", "delta_order")


def score_response(inst, text):
    # Ровно то, что делает evaluator.evaluate: resps -> фильтры задачи -> task.process_results(doc, [filtered_resps]).
    inst.resps = [text]
    inst.filtered_resps = {}
    for filter_ensemble in task._filters:
        filter_ensemble.apply([inst])
    scores, preds = {}, {}
    for name in FILTER_NAMES:
        assert name in inst.filtered_resps, "нет результата фильтра " + name + ": " + str(list(inst.filtered_resps))
        pred = inst.filtered_resps[name]
        assert isinstance(pred, str), "ожидалась строка после фильтра " + name + ", получено " + type(pred).__name__ + ": " + repr(pred)
        metrics = task.process_results(inst.doc, [pred])
        assert "exact_match" in metrics, "process_results не вернул exact_match: " + str(metrics)
        value = float(metrics["exact_match"])
        assert value in (0.0, 1.0), "exact_match должен быть 0 или 1, получено " + str(value)
        scores[name] = value
        preds[name] = pred
    return scores, preds


def decode_generation(gen_token_ids):
    text_raw = lm.tok_decode(gen_token_ids)
    return postprocess_generated_text(generation=text_raw, stop=UNTIL, think_end_token=None)


def run_generation(inst, remasking, num_steps, gen_length=None, block_length=None):
    gen_length = GEN_LENGTH if gen_length is None else gen_length
    block_length = BLOCK_LENGTH if block_length is None else block_length
    prompt = torch.tensor([PROMPT_IDS[inst.doc_id]], dtype=torch.long, device=model.device)
    torch.cuda.synchronize()
    t0 = time.time()
    x, steplog = llada_generate_logged(
        model, prompt, num_steps=num_steps, gen_length=gen_length, block_length=block_length,
        temperature=TEMPERATURE, cfg_scale=0.0, remasking=remasking, mask_id=MASK_ID, use_pc=False)
    torch.cuda.synchronize()
    return x, steplog, prompt.shape[1], time.time() - t0


def derive_seed(seed, task_id):
    # Совпадает с task_seed в exp3; правило/число шагов не входят в производную.
    digest = hashlib.sha256(f"exp3:{seed}:{task_id}".encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "little")


def seed_everything(value):
    random.seed(value)
    np.random.seed(value)
    torch.manual_seed(value)
    torch.cuda.manual_seed_all(value)


def process_task(inst, remasking, num_steps):
    task_id = inst.doc_id
    derived = derive_seed(SEED_USED, task_id)
    seed_everything(derived)
    x, steplog, prompt_len, sec = run_generation(inst, remasking, num_steps)
    gen_ids = x[0, prompt_len:].tolist()
    assert len(gen_ids) == GEN_LENGTH
    assert MASK_ID not in gen_ids, "задача " + str(task_id) + ": в ответе остались маски (mask_id=" + str(MASK_ID) + ") - несостыковка, остановка"
    assert len(steplog) == num_steps, "задача " + str(task_id) + ": в журнале " + str(len(steplog)) + " шагов вместо " + str(num_steps)
    text = decode_generation(gen_ids)
    scores, preds = score_response(inst, text)
    row = {
        "task_id": task_id, "seed": SEED_USED, "task_seed": derived, "correct": int(scores["flexible-extract"] == 1.0), "prediction": preds["flexible-extract"],
        "num_steps": num_steps, "remasking": remasking, "pc_active_steps": 0, "delta_words": None, "delta_order": None,
        # --- дополнительные поля (не входят в обязательный формат) ---
        "correct_strict": int(scores["strict-match"] == 1.0), "prediction_strict": preds["strict-match"],
        "gold": gold_final(inst.doc["answer"]), "generation": text, "gen_length": GEN_LENGTH, "prompt_tokens": prompt_len, "sec": round(sec, 3),
    }
    step_rows = []
    for rec in steplog:
        d = {"task_id": task_id, "seed": SEED_USED, "task_seed": derived, "remasking": remasking, "num_steps": num_steps}
        d.update(rec)
        d.update({"delta_words": None, "delta_order": None, "pc_active_steps": 0})
        step_rows.append(d)
    return row, step_rows


def result_path(remasking, num_steps):
    return os.path.join(OUT_DIR, remasking + "_steps" + str(num_steps) + "_seed" + str(SEED_USED) + ".jsonl")


def steplog_path(remasking, num_steps):
    return os.path.join(OUT_DIR, "steplog_" + remasking + "_steps" + str(num_steps) + "_seed" + str(SEED_USED) + ".jsonl")


def read_jsonl_tolerant(path):
    # Читает только полные строки (оканчивающиеся на \n). Оборванный хвост (обрыв во время записи) отбрасывается и физически
    # обрезается; повреждённая ПОЛНАЯ строка - ошибка (автоматически не чиним).
    if not os.path.exists(path):
        return []
    with open(path, "rb") as f:
        data = f.read()
    cut = data.rfind(b"\n") + 1
    tail = data[cut:]
    if tail:
        if tail.strip():
            log("ВНИМАНИЕ: " + path + ": оборванная последняя строка (" + str(len(tail)) + " байт) отброшена")
        with open(path, "r+b") as f:
            f.truncate(cut)
            f.flush()
            os.fsync(f.fileno())
    rows = []
    for line_no, raw in enumerate(data[:cut].split(b"\n"), start=1):
        if not raw.strip():
            continue
        try:
            rows.append(json.loads(raw.decode("utf-8")))
        except Exception as e:
            raise RuntimeError(path + ": строка " + str(line_no) + " повреждена (" + type(e).__name__ + ": " + str(e) + "). Исправьте или удалите файл вручную.") from e
    return rows


def append_jsonl(path, records):
    payload = "".join(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n" for r in records)
    with open(path, "a", encoding="utf-8") as f:
        f.write(payload)
        f.flush()
        os.fsync(f.fileno())


def repair_steplog(path, done_ids):
    rows = read_jsonl_tolerant(path)
    keep = [r for r in rows if r["task_id"] in done_ids]
    if len(keep) != len(rows):
        dropped = sorted({r["task_id"] for r in rows if r["task_id"] not in done_ids})
        log("steplog " + os.path.basename(path) + ": удалены строки незавершённых задач " + str(dropped))
        tmp = path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            for r in keep:
                f.write(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n")
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, path)
    return keep


def validate_result_row(row, remasking, num_steps):
    missing = [k for k in REQUIRED_ROW_KEYS if k not in row]
    assert not missing, "в строке результата нет полей " + str(missing) + ": " + str(row.get("task_id"))
    assert row["remasking"] == remasking and row["num_steps"] == num_steps, "строка не из этой комбинации: " + str(row["remasking"]) + "/" + str(row["num_steps"])
    assert row["seed"] == SEED_USED and row["pc_active_steps"] == 0 and row["delta_words"] is None and row["delta_order"] is None
    assert row["correct"] in (0, 1) and isinstance(row["prediction"], str) and isinstance(row["task_id"], int)
    assert row.get("gen_length", GEN_LENGTH) == GEN_LENGTH, "в файле результаты с gen_length=" + str(row.get("gen_length")) + ", а GEN_LENGTH=" + str(GEN_LENGTH)


def run_combo(remasking, num_steps, instances_to_run, deadline=None):
    # Возвращает "done", "stopped" (deadline = MAX_RUNTIME_MIN) или "signal" (получен SIGTERM/SIGINT). Остановка - только ПОСЛЕ
    # полностью записанной задачи (steplog, затем результат, fsync), поэтому кэш всегда согласован.
    rpath, spath = result_path(remasking, num_steps), steplog_path(remasking, num_steps)
    rows = read_jsonl_tolerant(rpath)
    for r in rows:
        validate_result_row(r, remasking, num_steps)
    ids = [r["task_id"] for r in rows]
    assert len(ids) == len(set(ids)), rpath + ": повторяющиеся task_id"
    done = set(ids)
    repair_steplog(spath, done)
    todo = [inst for inst in instances_to_run if inst.doc_id not in done]
    log("[" + remasking + ", steps=" + str(num_steps) + "] уже готово " + str(len(done)) + ", осталось " + str(len(todo)))
    for k, inst in enumerate(todo, 1):
        row, step_rows = process_task(inst, remasking, num_steps)
        append_jsonl(spath, step_rows)    # 1) журнал шагов
        append_jsonl(rpath, [row])        # 2) строка результата = маркер завершения задачи
        torch.cuda.empty_cache()          # между задачами
        log("[" + remasking + "/" + str(num_steps) + "] задача " + str(inst.doc_id) + " (" + str(k) + "/" + str(len(todo)) + ") correct=" + str(row["correct"]) + " pred=" + repr(row["prediction"]) + " gold=" + repr(row["gold"]) + " " + format(row["sec"], ".1f") + " c")
        if STOP_REQUESTED["flag"]:
            log("[" + remasking + ", steps=" + str(num_steps) + "] остановка по сигналу после задачи " + str(inst.doc_id))
            return "signal"
        if deadline is not None and time.time() >= deadline:
            log("[" + remasking + ", steps=" + str(num_steps) + "] остановка по MAX_RUNTIME_MIN после задачи " + str(inst.doc_id))
            return "stopped"
    return "done"


def validate_combo(remasking, num_steps, expected_ids):
    rows = read_jsonl_tolerant(result_path(remasking, num_steps))
    for r in rows:
        validate_result_row(r, remasking, num_steps)
    ids = [r["task_id"] for r in rows]
    assert sorted(ids) == sorted(expected_ids), remasking + "/" + str(num_steps) + ": ожидались task_id " + str(len(expected_ids)) + " шт., в файле " + str(len(ids)) + " (повторы или пропуски)"
    steps_rows = read_jsonl_tolerant(steplog_path(remasking, num_steps))
    per_task = {}
    for s in steps_rows:
        per_task.setdefault(s["task_id"], []).append(s)
    assert set(per_task) == set(ids), remasking + "/" + str(num_steps) + ": steplog и результаты содержат разные task_id"
    for tid, recs in per_task.items():
        assert len(recs) == num_steps, remasking + "/" + str(num_steps) + ", задача " + str(tid) + ": в steplog " + str(len(recs)) + " шагов вместо " + str(num_steps)
        assert [r["global_step"] for r in recs] == list(range(num_steps)), remasking + "/" + str(num_steps) + ", задача " + str(tid) + ": нарушен порядок шагов"
    return rows


# ===================== 7. ДЫМОВОЙ ТЕСТ =====================
def assert_same(name_a, a, name_b, b, prompt_len, where, hint=""):
    if a.shape == b.shape and bool((a == b).all()):
        return
    if a.shape == b.shape:
        diff_pos = (a != b).nonzero()[:, 1].tolist()
        first = diff_pos[0]
        detail = ("позиций отличается: " + str(len(diff_pos)) + "; первая: " + str(first) + " (позиция " + str(first - prompt_len) + " в ответе); токены "
                  + str(a[0, first].item()) + " (" + name_a + ") и " + str(b[0, first].item()) + " (" + name_b + ")")
    else:
        detail = "формы различаются: " + str(tuple(a.shape)) + " и " + str(tuple(b.shape))
    raise AssertionError("РАСХОЖДЕНИЕ [" + where + "]: " + name_a + " != " + name_b + ". " + detail + ". " + hint + "Остановка: полный прогон запрещён, пока причина не найдена.")


def check_steplog(steplog, x, prompt_len, gen_length, block_length, num_steps, where):
    num_blocks = gen_length // block_length
    steps = num_steps // num_blocks
    assert len(steplog) == num_steps, "[" + where + "] в журнале " + str(len(steplog)) + " шагов вместо " + str(num_steps)
    seen = set()
    revealed_in_block = {}
    for k, rec in enumerate(steplog):
        b, i = divmod(k, steps)
        assert (rec["block"], rec["step_in_block"], rec["global_step"]) == (b, i, k), "[" + where + "] нарушен порядок шагов на " + str(k)
        start, end = prompt_len + b * block_length, prompt_len + (b + 1) * block_length
        expected_n = block_length // steps + (1 if i < block_length % steps else 0)
        pos = rec["positions_abs"]
        assert len(pos) == expected_n, "[" + where + "] шаг " + str(k) + ": раскрыто " + str(len(pos)) + " позиций вместо " + str(expected_n)
        assert all(start <= p < end for p in pos), "[" + where + "] шаг " + str(k) + ": позиции вне текущего блока [" + str(start) + ", " + str(end) + "): " + str(pos)
        assert not (set(pos) & seen), "[" + where + "] шаг " + str(k) + ": позиция раскрыта повторно"
        seen |= set(pos)
        assert rec["positions_rel"] == [p - prompt_len for p in pos], "[" + where + "] шаг " + str(k) + ": positions_rel не согласованы"
        assert rec["token_ids"] == [x[0, p].item() for p in pos], "[" + where + "] шаг " + str(k) + ": токены журнала не совпали с результатом"
        done_before = revealed_in_block.get(b, 0)
        assert rec["n_masked_block_before"] == block_length - done_before, "[" + where + "] шаг " + str(k) + ": неверное число масок в блоке"
        revealed_in_block[b] = done_before + len(pos)
        conf = rec["confidence"]
        assert all(math.isfinite(c) and -1e-2 <= c <= 1 + 1e-2 for c in conf), "[" + where + "] шаг " + str(k) + ": уверенность вне [0, 1] или не конечна: " + str(conf)
        assert all(conf[t] >= conf[t + 1] for t in range(len(conf) - 1)), "[" + where + "] шаг " + str(k) + ": уверенность не убывает внутри шага"
    assert seen == set(range(prompt_len, prompt_len + gen_length)), "[" + where + "] раскрыты не все позиции ответа"


def run_smoke(gpu_index):
    import io
    import contextlib
    smoke_n, smoke_steps, smoke_gen, smoke_block = 3, 8, 64, 32
    assert smoke_gen % smoke_block == 0 and smoke_steps % (smoke_gen // smoke_block) == 0
    smoke_insts = INSTANCES[:smoke_n]

    log("A. Проверка пути оценки на синтетических ответах")
    for inst in smoke_insts:
        gold_text = inst.doc["answer"]
        s_ok, p_ok = score_response(inst, gold_text)
        assert s_ok["flexible-extract"] == 1.0 and s_ok["strict-match"] == 1.0, "эталонный ответ задачи " + str(inst.doc_id) + " оценён как неверный: " + str(p_ok) + " - путь оценки сломан"
        gold_num = gold_final(gold_text).replace(",", "")
        try:
            wrong_num = str(int(float(gold_num)) + 1)
        except ValueError:
            log("   задача " + str(inst.doc_id) + ": эталон не число, проверка искажённого ответа пропущена")
            continue
        wrong_text = gold_text.rsplit("####", 1)[0] + "#### " + wrong_num
        s_bad, p_bad = score_response(inst, wrong_text)
        assert s_bad["flexible-extract"] == 0.0 and s_bad["strict-match"] == 0.0, "искажённый ответ задачи " + str(inst.doc_id) + " оценён как верный: " + str(p_bad) + " - путь оценки сломан"
        log("   задача " + str(inst.doc_id) + ": эталон -> верно (" + repr(p_ok["flexible-extract"]) + "); искажённый -> неверно (" + repr(p_bad["flexible-extract"]) + ")")

    log("B. Проверка промпта")
    for inst in smoke_insts:
        ctx_text = inst.args[0]
        assert SYSTEM_INSTRUCTION in ctx_text, "в промпте задачи " + str(inst.doc_id) + " нет system-инструкции"
        assert inst.doc["question"] in ctx_text, "в промпте задачи " + str(inst.doc_id) + " нет текста вопроса"
        manual = lm.apply_chat_template([{"role": "system", "content": SYSTEM_INSTRUCTION}, {"role": "user", "content": task.doc_to_text(inst.doc)}], add_generation_prompt=True)
        log("   задача " + str(inst.doc_id) + ": совпадает с ручной сборкой system+user через chat template: " + str(manual == ctx_text))

    def quiet_call(fn, *a, **kw):
        with contextlib.redirect_stderr(io.StringIO()):
            return fn(*a, **kw)

    log("C0. Логиты конечны (защита от NaN/inf; особенно важно при DTYPE=float16)")
    for inst in smoke_insts:
        p_len = len(PROMPT_IDS[inst.doc_id])
        probe = torch.full((1, p_len + smoke_gen), MASK_ID, dtype=torch.long, device=model.device)
        probe[0, :p_len] = torch.tensor(PROMPT_IDS[inst.doc_id], dtype=torch.long, device=model.device)
        with torch.no_grad():
            lg = model(probe).logits
        assert bool(torch.isfinite(lg).all()), "В логитах задачи " + str(inst.doc_id) + " есть NaN/inf при DTYPE=" + DTYPE + ". Остановка."
        log("   задача " + str(inst.doc_id) + ": логиты конечны, форма " + str(tuple(lg.shape)) + ", dtype " + str(lg.dtype))
        del probe, lg
        torch.cuda.empty_cache()

    log("C. Токены: копия против оригинала (GPU)")
    samples = {}
    for remasking in REMASKING_LIST:
        for inst in smoke_insts:
            prompt = torch.tensor([PROMPT_IDS[inst.doc_id]], dtype=torch.long, device=model.device)
            P = prompt.shape[1]
            common = dict(num_steps=smoke_steps, gen_length=smoke_gen, block_length=smoke_block, temperature=TEMPERATURE, cfg_scale=0., remasking=remasking, mask_id=MASK_ID)
            where = "remasking=" + remasking + ", задача " + str(inst.doc_id)
            x_copy, log_copy = llada_generate_logged(model, prompt, use_pc=False, **common)
            x_copy_2, _ = llada_generate_logged(model, prompt, use_pc=False, **common)
            x_copy_plain, _ = llada_generate_logged(PLAIN, prompt, use_pc=False, **common)
            with torch.no_grad():
                x_orig_plain = quiet_call(llada_diffusion_generate, PLAIN, prompt, **common)
                x_orig_codd = quiet_call(llada_diffusion_generate, model, prompt, use_pc=False, **common)
            assert_same("копия (прогон 2)", x_copy_2, "копия (прогон 1)", x_copy, P, where, hint="Копия расходится сама с собой: недетерминизм GPU/ядер, а не ошибка логики. ")
            assert_same("копия на PlainView", x_copy_plain, "копия на CoddLlada", x_copy, P, where)
            assert_same("копия на PlainView", x_copy_plain, "оригинал на PlainView", x_orig_plain, P, where)
            assert_same("копия на CoddLlada", x_copy, "оригинал на CoddLlada(use_pc=False)", x_orig_codd, P, where)
            assert bool((x_copy[:, :P] == prompt).all()), "[" + where + "] промпт в результате изменился"
            assert not bool((x_copy[:, P:] == MASK_ID).any()), "[" + where + "] в ответе остались маски"
            check_steplog(log_copy, x_copy, P, smoke_gen, smoke_block, smoke_steps, where)
            samples[(remasking, inst.doc_id)] = tokenizer.decode(x_copy[0, P:].tolist(), skip_special_tokens=True)
            log("   OK [" + where + "]: 5 прогонов совпали по токенам, журнал согласован")
            torch.cuda.empty_cache()
    log("Пример ответа дымового теста (задача 0, " + REMASKING_LIST[0] + ", 8 шагов, 64 токена; текст может быть бессмысленным - шагов мало):")
    log(repr(samples[(REMASKING_LIST[0], 0)][:300]))
    log("ДЫМОВОЙ ТЕСТ (корректность): ПРОЙДЕН")

    # --- пиковая видеопамять при РЕАЛЬНОМ GEN_LENGTH на самой длинной по токенам задаче ---
    gib = 1024 ** 3
    longest = max(INSTANCES, key=lambda i: len(PROMPT_IDS[i.doc_id]))
    total_mem = torch.cuda.get_device_properties(gpu_index).total_memory
    peak_report = {}
    for rm in REMASKING_LIST:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        run_generation(longest, rm, NUM_BLOCKS)
        peak_alloc = torch.cuda.max_memory_allocated()
        peak_res = torch.cuda.max_memory_reserved()
        peak_report[rm] = {"peak_allocated_gib": peak_alloc / gib, "peak_reserved_gib": peak_res / gib, "headroom_gib": (total_mem - peak_alloc) / gib}
        log("Пик памяти [" + rm + "], самая длинная задача (" + str(len(PROMPT_IDS[longest.doc_id])) + " ток. промпта + " + str(GEN_LENGTH) + " ответа): allocated " + format(peak_alloc / gib, ".2f")
            + " ГиБ, reserved " + format(peak_res / gib, ".2f") + " ГиБ из " + format(total_mem / gib, ".2f") + " ГиБ; запас (total - allocated) " + format((total_mem - peak_alloc) / gib, ".2f") + " ГиБ")
        assert total_mem - peak_alloc >= 2 * gib, "Запас видеопамяти " + format((total_mem - peak_alloc) / gib, ".2f") + " ГиБ < 2 ГиБ при remasking=" + rm + ", GEN_LENGTH=" + str(GEN_LENGTH) + ", DTYPE=" + DTYPE
        if total_mem - peak_res < 2 * gib:
            log("   ВНИМАНИЕ: с учётом кэша аллокатора (reserved) запас " + format((total_mem - peak_res) / gib, ".2f") + " ГиБ < 2 ГиБ; empty_cache между задачами включён.")
        torch.cuda.empty_cache()

    # --- секунды на задачу и прогноз ---
    timing_n = 2
    timing_remasking = REMASKING_LIST[0]
    timing_steps = NUM_STEPS_LIST[0]
    secs = []
    for inst in INSTANCES[:timing_n]:
        _x, _log, _plen, sec = run_generation(inst, timing_remasking, timing_steps)
        secs.append(sec)
        torch.cuda.empty_cache()
    sec_per_task = sum(secs) / len(secs)
    sec_per_forward = sec_per_task / timing_steps
    log("Замер: " + str(timing_n) + " задачи, remasking=" + timing_remasking + ", num_steps=" + str(timing_steps) + ", GEN_LENGTH=" + str(GEN_LENGTH))
    log("Секунд на задачу: " + format(sec_per_task, ".2f") + " (" + str(timing_steps) + " шагов), на один проход модели: " + format(sec_per_forward, ".3f") + " c")
    n_eff = RUN_LIMIT if RUN_LIMIT is not None else N_TASKS
    per_combo = {}
    log("Прогноз времени (пропорционально числу проходов модели; оценка, НЕ гарантия; задач на комбинацию: " + str(n_eff) + "):")
    for rm in REMASKING_LIST:
        for st in NUM_STEPS_LIST:
            per_combo[(rm, st)] = sec_per_forward * st * n_eff
            log("   " + rm.ljust(16) + " steps=" + str(st).ljust(4) + " ~" + format(sec_per_forward * st, "6.1f") + " c/задачу, ~" + str(datetime.timedelta(seconds=int(per_combo[(rm, st)]))) + " на комбинацию")
    total_forecast = sum(per_combo.values())
    log("ИТОГО на " + str(len(per_combo)) + " комбинации x " + str(n_eff) + " задач: ~" + str(datetime.timedelta(seconds=int(total_forecast))) + " (~" + format(total_forecast / 3600, ".1f") + " ч)")
    if MAX_RUNTIME_MIN is not None:
        log("MAX_RUNTIME_MIN=" + str(MAX_RUNTIME_MIN) + ": за один запуск будет сделано примерно " + format(MAX_RUNTIME_MIN * 60 / total_forecast * 100, ".0f") + "% полного прогона (оценка).")
    write_json_atomic(SMOKE_PATH, {
        "ok": True, "finished_utc": utc_now(), "exp_sha256": EXP_SHA256, "gen_length": GEN_LENGTH,
        "sec_per_task_first_steps": sec_per_task, "sec_per_forward": sec_per_forward,
        "forecast_seconds_total": total_forecast, "forecast_tasks_per_combo": n_eff,
        "pyjuice_mode": PYJUICE_MODE, "dtype": DTYPE, "gpu_name": ENV_INFO.get("gpu_name"), "gpu_capability": ENV_INFO.get("gpu_capability"),
        "peak_memory": peak_report, "total_gpu_memory_gib": total_mem / gib,
    })
    log("smoke.json записан: полный прогон разрешён.")


# ===================== 8. ПОЛНЫЙ ПРОГОН =====================
def check_smoke_done():
    if not REQUIRE_SMOKE:
        log("REQUIRE_SMOKE=false: проверка smoke.json пропущена")
        return
    assert os.path.exists(SMOKE_PATH), "Нет " + SMOKE_PATH + ": сначала запустите режим smoke и дождитесь успешного завершения"
    with open(SMOKE_PATH, encoding="utf-8") as f:
        sm = json.load(f)
    assert sm.get("ok") is True, "smoke.json: ok != true"
    assert sm.get("exp_sha256") == EXP_SHA256, "smoke.json получен при другом конфиге (exp_sha256 не совпал): повторите smoke"
    assert sm.get("dtype") == DTYPE and sm.get("gen_length") == GEN_LENGTH, "smoke.json получен при другом dtype/GEN_LENGTH: повторите smoke"
    log("smoke.json: успешный дымовой тест от " + str(sm.get("finished_utc")) + " найден")


def run_full():
    run_instances = INSTANCES if RUN_LIMIT is None else INSTANCES[:RUN_LIMIT]
    if RUN_LIMIT is not None:
        log("=" * 90)
        log("ВНИМАНИЕ: RUN_LIMIT=" + str(RUN_LIMIT) + ". Будут обработаны только первые " + str(RUN_LIMIT) + " задач из " + str(N_TASKS) + ". ПРОГОН НЕПОЛНЫЙ.")
        log("=" * 90)
    log("GEN_LENGTH=" + str(GEN_LENGTH) + ", BLOCK_LENGTH=" + str(BLOCK_LENGTH) + ", задач: " + str(len(run_instances)) + ", комбинаций: " + str(len(REMASKING_LIST) * len(NUM_STEPS_LIST)))
    t_all = time.time()
    deadline = None if MAX_RUNTIME_MIN is None else t_all + 60 * MAX_RUNTIME_MIN
    if deadline is not None:
        log("MAX_RUNTIME_MIN=" + str(MAX_RUNTIME_MIN) + ": после завершённой задачи, на которой истечёт лимит, прогон корректно остановится.")
    stopped = None
    for remasking in REMASKING_LIST:
        for num_steps in NUM_STEPS_LIST:
            if STOP_REQUESTED["flag"]:
                stopped = "signal"
                break
            if deadline is not None and time.time() >= deadline:
                stopped = "stopped"
                break
            status = run_combo(remasking, num_steps, run_instances, deadline=deadline)
            if status != "done":
                stopped = status
                break
        if stopped:
            break
    log("Цикл прогона завершён за " + str(datetime.timedelta(seconds=int(time.time() - t_all))) + " (в этом запуске), остановка: " + str(stopped))

    for remasking in REMASKING_LIST:
        for num_steps in NUM_STEPS_LIST:
            present = sorted(r["task_id"] for r in read_jsonl_tolerant(result_path(remasking, num_steps)))
            if stopped:
                validate_combo(remasking, num_steps, present)      # прогон неполный: проверяем только согласованность готового
            elif RUN_LIMIT is None:
                validate_combo(remasking, num_steps, TASK_IDS)
            else:
                wanted = [inst.doc_id for inst in run_instances]
                assert set(wanted) <= set(present), remasking + "/" + str(num_steps) + ": нет результатов для задач " + str(sorted(set(wanted) - set(present)))
                validate_combo(remasking, num_steps, present)
            log("   проверено: " + os.path.basename(result_path(remasking, num_steps)) + ": " + str(len(present)) + " задач, steplog согласован")
    if stopped:
        return "STOPPED", "остановлено (" + stopped + "); чтобы продолжить - запустите режим full ещё раз, готовые задачи будут пропущены"
    if RUN_LIMIT is None:
        return "DONE", "все " + str(len(REMASKING_LIST) * len(NUM_STEPS_LIST)) + " комбинации по " + str(N_TASKS) + " задач"
    return "DONE", "НЕПОЛНЫЙ прогон (RUN_LIMIT=" + str(RUN_LIMIT) + ")"


# ===================== 9. СВОДКА =====================
def bootstrap_ci(values, n_boot=None, seed=None, alpha=0.05):
    # Percentile bootstrap по задачам: ресэмплируем задачи с возвратом, считаем долю верных.
    n_boot = N_BOOTSTRAP if n_boot is None else n_boot
    seed = BOOTSTRAP_SEED if seed is None else seed
    v = np.asarray(values, dtype=np.float64)
    assert v.ndim == 1 and v.size > 0, "пустой набор значений"
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, v.size, size=(n_boot, v.size))
    means = v[idx].mean(axis=1)
    lo, hi = np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(v.mean()), float(lo), float(hi)


def run_summary():
    os.makedirs(OUT_DIR, exist_ok=True)
    ids_sha = None
    if os.path.exists(TASKS_PATH):
        with open(TASKS_PATH, encoding="utf-8") as f:
            ids_sha = json.load(f).get("ids_sha256")
    rows_out = []
    for remasking in REMASKING_LIST:
        for num_steps in NUM_STEPS_LIST:
            rows = sorted(read_jsonl_tolerant(result_path(remasking, num_steps)), key=lambda r: r["task_id"])
            if not rows:
                log("нет результатов для " + remasking + "/" + str(num_steps))
                continue
            acc, lo, hi = bootstrap_ci([r["correct"] for r in rows])
            acc_s, lo_s, hi_s = bootstrap_ci([r["correct_strict"] for r in rows])
            rows_out.append({"remasking": remasking, "num_steps": num_steps, "n": len(rows), "accuracy": acc, "ci_low": lo, "ci_high": hi,
                             "accuracy_strict": acc_s, "ci_low_strict": lo_s, "ci_high_strict": hi_s})
    log("exp1 (обычная LLaDA, HMM не используется) | GSM8K test, первые " + str(N_TASKS) + " задач | GEN_LENGTH=" + str(GEN_LENGTH) + ", BLOCK_LENGTH=" + str(BLOCK_LENGTH) + ", greedy | bootstrap: " + str(N_BOOTSTRAP) + " повторов, 95%")
    log("sha256 списка id задач: " + str(ids_sha))
    header = "remasking".ljust(16) + "steps".rjust(6) + "n".rjust(5) + "   " + "accuracy (flexible-extract)".ljust(30) + "accuracy (strict-match)"
    print(header, flush=True)
    print("-" * (len(header) + 6), flush=True)
    for s in rows_out:
        flag = "" if s["n"] == N_TASKS else "  <- НЕПОЛНО"
        a = format(100 * s["accuracy"], "5.1f") + "% [" + format(100 * s["ci_low"], "5.1f") + "; " + format(100 * s["ci_high"], "5.1f") + "]"
        b = format(100 * s["accuracy_strict"], "5.1f") + "% [" + format(100 * s["ci_low_strict"], "5.1f") + "; " + format(100 * s["ci_high_strict"], "5.1f") + "]"
        print(s["remasking"].ljust(16) + str(s["num_steps"]).rjust(6) + str(s["n"]).rjust(5) + "   " + a.ljust(30) + b + flag, flush=True)
    print("Доверительный интервал отражает только выбор задач (разброс по задачам), а не случайность генерации: случайности нет.", flush=True)
    complete = (len(rows_out) == len(REMASKING_LIST) * len(NUM_STEPS_LIST)) and all(s["n"] == N_TASKS for s in rows_out)
    write_json_atomic(SUMMARY_PATH, {
        "experiment": EXP_NAME, "gen_length": GEN_LENGTH, "block_length": BLOCK_LENGTH, "n_tasks": N_TASKS, "ids_sha256": ids_sha,
        "n_bootstrap": N_BOOTSTRAP, "bootstrap_seed": BOOTSTRAP_SEED, "complete": complete, "rows": rows_out, "updated_utc": utc_now(),
    })
    log("summary.json записан: " + SUMMARY_PATH + " | полный: " + str(complete))
    return complete


# ===================== MAIN =====================
def main():
    if MODE == "summary":
        set_status("running", "summary")
        run_summary()
        set_status("done", "summary")
        return 0
    set_status("running", "запуск режима " + MODE)
    if MODE == "full":
        signal.signal(signal.SIGTERM, _on_stop_signal)
        signal.signal(signal.SIGINT, _on_stop_signal)
    gpu_index, _total = check_environment()
    prepare_pyjuice()
    import_repo_code()
    write_env_and_config()
    if MODE == "full":
        check_smoke_done()
    load_model()
    check_model_code()
    setup_tasks()
    if MODE == "smoke":
        run_smoke(gpu_index)
        set_status("done", "smoke пройден")
        return 0
    state, message = run_full()
    run_summary()
    set_status(state.lower(), message)
    return 0


if __name__ == "__main__":
    try:
        sys.exit(main())
    except SystemExit:
        raise
    except BaseException as exc:
        import traceback
        traceback.print_exc()
        try:
            set_status("failed", type(exc).__name__ + ": " + str(exc)[:500])
        except Exception:
            pass
        sys.exit(1)
"""


def write_run_files():
    Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
    Path(WORK_DIR, "logs").mkdir(parents=True, exist_ok=True)
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
    core_path = Path(WORK_DIR, "exp1_core.py")
    core_path.write_text(CORE_SOURCE, encoding="utf-8")
    cfg_path = Path(WORK_DIR, "config.json")
    cfg_path.write_text(json.dumps(CORE_CONFIG, ensure_ascii=False, indent=2, sort_keys=True), encoding="utf-8")
    return core_path, cfg_path


_core_path, _cfg_path = write_run_files()
print("записан:", _core_path, "(%d строк, sha256 %s)" % (CORE_SOURCE.count("\n"), hashlib.sha256(CORE_SOURCE.encode("utf-8")).hexdigest()[:16]))
print("записан:", _cfg_path)
_chk = subprocess.run([VENV_PYTHON] + PROCESS_PYTHON_FLAGS + ["-c", "import ast,sys; ast.parse(open(sys.argv[1], encoding='utf-8').read()); print('синтаксис exp1_core.py (Python ядра): OK')", str(_core_path)],
                      stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
print(_chk.stdout.strip() or _chk.stderr.strip())
assert _chk.returncode == 0, "exp1_core.py не проходит синтаксическую проверку Python ядра"

## 3. Управление процессом и мониторинг

Функции: `start_core(mode)` (запуск через `subprocess.Popen(start_new_session=True)`, вывод в `exp1_run/logs/exp1_<mode>_<время>.log`, pid в `exp1_run/exp1.pid`), `status()`, `monitor()` (цикл с `sleep`: ядро остаётся занятым, поэтому автоостановка простаивающей ВМ не срабатывает), `stop_core()` (SIGTERM группе процессов; процесс full дописывает кэш и останавливается после текущей задачи, затем SIGKILL по таймауту).

`monitor()` можно запускать и после перезапуска ядра: он найдёт процесс по pid-файлу. Остановить мониторинг (процесс продолжит работу): **Interrupt kernel**.

In [ ]:
# ===================== 3. УПРАВЛЕНИЕ ПРОЦЕССОМ И МОНИТОРИНГ =====================
PID_FILE = Path(WORK_DIR, "exp1.pid")
STATE_FILE = Path(WORK_DIR, "exp1_state.json")
LOG_DIR = Path(WORK_DIR, "logs")
CORE_PROC = {"popen": None}      # объект Popen живёт только в этом ядре (нужен, чтобы забирать код выхода и не оставлять зомби)
_MON_HIST = []


def _utc_now():
    return datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


def read_state():
    try:
        return json.loads(STATE_FILE.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return None


def pid_alive(pid):
    # Процесс жив, если он существует, не зомби и в его командной строке есть exp1_core.py (защита от переиспользования pid).
    popen = CORE_PROC.get("popen")
    if popen is not None and popen.pid == pid:
        return popen.poll() is None
    try:
        os.kill(pid, 0)
    except OSError:
        return False
    try:
        cmdline = Path("/proc/%d/cmdline" % pid).read_bytes()
        stat = Path("/proc/%d/stat" % pid).read_text()
    except OSError:
        return False
    state = stat.rsplit(")", 1)[-1].split()[0] if ")" in stat else "?"
    return b"exp1_core.py" in cmdline and state != "Z"


def current_process():
    # Возвращает (pid, state) живого процесса или (None, state_or_None)
    state = read_state()
    pid = None
    if PID_FILE.exists():
        try:
            pid = int(PID_FILE.read_text().strip())
        except ValueError:
            pid = None
    if pid is not None and pid_alive(pid):
        return pid, state
    return None, state


def start_core(mode):
    assert mode in ("smoke", "full", "summary"), mode
    pid, _state = current_process()
    if pid is not None:
        raise RuntimeError("Процесс уже запущен (pid %d). Дождитесь его окончания или вызовите stop_core()." % pid)
    write_run_files()        # конфиг всегда актуальный
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = LOG_DIR / ("exp1_%s_%s.log" % (mode, stamp))
    cmd = ["env"] + PROCESS_ENV + [VENV_PYTHON] + PROCESS_PYTHON_FLAGS + ["exp1_core.py", "--mode", mode]
    logf = open(str(log_path), "ab")
    try:
        proc = subprocess.Popen(cmd, cwd=WORK_DIR, stdout=logf, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)
    finally:
        logf.close()
    CORE_PROC["popen"] = proc
    PID_FILE.write_text(str(proc.pid), encoding="utf-8")
    STATE_FILE.write_text(json.dumps({"pid": proc.pid, "mode": mode, "started_utc": _utc_now(), "log": str(log_path), "cmd": cmd}, ensure_ascii=False, indent=2), encoding="utf-8")
    print("запущен процесс pid=%d, режим %s" % (proc.pid, mode))
    print("команда:", " ".join(cmd))
    print("лог:", log_path)
    return proc.pid


def stop_core(timeout_s=240):
    pid, state = current_process()
    if pid is None:
        print("Живого процесса нет.")
        return False
    print("Останавливаю pid=%d (SIGTERM группе процессов). Режим full дописывает кэш и steplog и останавливается после текущей задачи." % pid)
    try:
        os.killpg(pid, signal.SIGTERM)
    except OSError as e:
        print("killpg:", e)
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if not pid_alive(pid):
            print("Процесс остановлен за %.0f c." % (time.time() - t0))
            return True
        time.sleep(3)
    print("Не остановился за %d c: SIGKILL (кэш согласован: незавершённая задача будет пропущена при возобновлении)." % timeout_s)
    try:
        os.killpg(pid, signal.SIGKILL)
    except OSError as e:
        print("killpg:", e)
    time.sleep(3)
    return not pid_alive(pid)


def tail_file(path, n):
    try:
        data = Path(path).read_bytes()[-20000:]
    except OSError:
        return []
    text = data.decode("utf-8", "replace").replace("\r", "\n")
    return [l for l in text.splitlines() if l.strip()][-n:]


def _read_rows(path):
    rows = []
    try:
        data = Path(path).read_bytes()
    except OSError:
        return rows
    cut = data.rfind(b"\n") + 1
    for raw in data[:cut].split(b"\n"):
        if raw.strip():
            try:
                rows.append(json.loads(raw.decode("utf-8")))
            except ValueError:
                pass
    return rows


def progress_snapshot():
    target = RUN_LIMIT if RUN_LIMIT is not None else N_TASKS
    snap = []
    for rm in REMASKING_LIST:
        for st in NUM_STEPS_LIST:
            rows = _read_rows(Path(OUT_DIR, "%s_steps%d_seed%d.jsonl" % (rm, st, SEED_USED)))
            secs = [r["sec"] for r in rows if isinstance(r.get("sec"), (int, float))]
            snap.append({"remasking": rm, "steps": st, "done": len(rows), "target": target,
                         "correct": sum(int(r.get("correct", 0)) for r in rows),
                         "mean_sec": (sum(secs) / len(secs)) if secs else None})
    return snap


def _fmt_td(seconds):
    return str(datetime.timedelta(seconds=int(max(0, seconds))))


def render_status(extra_header=""):
    pid, state = current_process()
    lines = []
    if extra_header:
        lines.append(extra_header)
    if state:
        lines.append("режим: %s | pid: %s | запущен: %s | %s" % (state.get("mode"), state.get("pid"), state.get("started_utc"), "ЖИВ" if pid is not None else "не работает"))
    else:
        lines.append("процесс ещё не запускался")
    try:
        st = json.loads(Path(OUT_DIR, "status.json").read_text(encoding="utf-8"))
        lines.append("статус процесса: %s %s (обновлён %s)" % (st.get("state"), st.get("message", ""), st.get("updated_utc")))
    except (OSError, ValueError):
        pass
    snap = progress_snapshot()
    total_done = sum(s["done"] for s in snap)
    total_target = sum(s["target"] for s in snap)
    if total_done or (state and state.get("mode") == "full"):
        lines.append("-" * 78)
        lines.append("%-16s %6s %12s %10s %12s" % ("remasking", "steps", "готово", "верно", "сек/задачу"))
        for s in snap:
            acc = ("%.0f%%" % (100.0 * s["correct"] / s["done"])) if s["done"] else "-"
            lines.append("%-16s %6d %12s %10s %12s" % (s["remasking"], s["steps"], "%d/%d" % (s["done"], s["target"]), acc, ("%.1f" % s["mean_sec"]) if s["mean_sec"] else "-"))
        now = time.time()
        _MON_HIST.append((now, total_done))
        while len(_MON_HIST) > 2 and now - _MON_HIST[0][0] > 900:
            _MON_HIST.pop(0)
        speed = None
        if len(_MON_HIST) >= 2 and _MON_HIST[-1][0] > _MON_HIST[0][0]:
            speed = (_MON_HIST[-1][1] - _MON_HIST[0][1]) / (_MON_HIST[-1][0] - _MON_HIST[0][0]) * 60.0
        # прогноз по среднему времени на задачу в каждой комбинации (для комбинаций без данных - по времени на шаг из других)
        per_step = [s["mean_sec"] / s["steps"] for s in snap if s["mean_sec"]]
        per_step_avg = (sum(per_step) / len(per_step)) if per_step else None
        remain = 0.0
        ok = True
        for s in snap:
            left = s["target"] - s["done"]
            sec = s["mean_sec"] if s["mean_sec"] else (per_step_avg * s["steps"] if per_step_avg else None)
            if left > 0:
                if sec is None:
                    ok = False
                else:
                    remain += left * sec
        lines.append("-" * 78)
        lines.append("всего %d/%d задач (%.0f%%)%s" % (total_done, total_target, 100.0 * total_done / max(1, total_target),
                                                       (" | скорость за последние минуты: %.2f задач/мин" % speed) if speed is not None else ""))
        if ok and total_done < total_target:
            lines.append("прогноз до конца по среднему времени задачи: ~%s (оценка)" % _fmt_td(remain))
        elif total_done >= total_target:
            lines.append("все задачи готовы")
    log_path = (state or {}).get("log")
    if not log_path:
        logs = sorted(LOG_DIR.glob("exp1_*.log"))
        log_path = str(logs[-1]) if logs else None
    if log_path:
        lines.append("-" * 78)
        lines.append("хвост лога (%s):" % log_path)
        lines.extend(tail_file(log_path, MONITOR_TAIL_LINES))
    return "\n".join(lines), pid


def status():
    text, _pid = render_status()
    print(text)


def monitor(interval=None, max_minutes=None):
    # Цикл с sleep: ядро остаётся занятым (ВМ не простаивает). Заканчивается, когда процесс завершился. Interrupt kernel - выйти, процесс продолжит работу.
    interval = MONITOR_INTERVAL_S if interval is None else interval
    try:
        from IPython.display import clear_output
    except Exception:
        clear_output = None
    t_start = time.time()
    try:
        while True:
            text, pid = render_status("[%s UTC] мониторинг, обновление каждые %d c (Interrupt kernel - выйти; процесс продолжит работу)" % (_utc_now(), interval))
            if clear_output is not None:
                clear_output(wait=True)
            print(text)
            if pid is None:
                print("\nПроцесс не работает. Мониторинг завершён.")
                break
            if max_minutes is not None and (time.time() - t_start) > max_minutes * 60:
                print("\nДостигнут лимит времени мониторинга; процесс продолжает работу.")
                break
            time.sleep(interval)
    except KeyboardInterrupt:
        print("\nМониторинг остановлен вручную. Процесс продолжает работать (status() / monitor() подключатся снова).")


_pid, _state = current_process()
print("процесс сейчас:", ("pid %d (режим %s)" % (_pid, (_state or {}).get("mode"))) if _pid else "не запущен")

## 4. Дымовой тест (`--mode smoke`)

3 задачи, 8 шагов, 64 токена: токены копии `llada_diffusion_generate` должны совпасть с оригиналом (базовая ветка и путь `CoddLlada(use_pc=False)`), журнал шагов согласован, логиты конечны; затем пик видеопамяти при реальном `GEN_LENGTH` (запас не менее 2 ГиБ), секунды на задачу и прогноз времени на 1 комбинацию x 300 задач. При любом расхождении процесс завершается ошибкой, `smoke.json` не пишется, и полный прогон не запустится. Загрузка модели занимает несколько минут.

In [ ]:
# ===================== 4. ДЫМОВОЙ ТЕСТ: запуск процесса и мониторинг =====================
start_core("smoke")
monitor()

## 5. Полный прогон (`--mode full`)

1 комбинация (`remasking` x `num_steps`) по `N_TASKS` задач, один детерминированный запуск на комбинацию (`seed0`). Требуется успешный дымовой тест (`results/exp1_llada_default/smoke.json` при том же конфиге). Прогон **идемпотентен**: после обрыва сессии DataSphere просто запустите эту ячейку снова (после перезапуска ядра сначала выполните ячейки 0-3); готовые задачи пропускаются. `RUN_LIMIT` и `MAX_RUNTIME_MIN` задаются в конфиге; при `RUN_LIMIT != None` прогон помечается как неполный. По завершении процесс сам считает сводку (`summary.json`).

In [ ]:
# ===================== 5. ПОЛНЫЙ ПРОГОН: запуск процесса и мониторинг =====================
start_core("full")
monitor()

### Подключиться к уже идущему процессу / состояние / остановить

- `monitor()` - подключиться к процессу после перезапуска ядра (ядро снова занято, ВМ не простаивает);
- `status()` - разовый снимок без цикла;
- `stop_core()` - остановить процесс (для `full` корректно: после текущей задачи). После остановки можно продолжить, запустив ячейку полного прогона снова.

In [ ]:
# ===================== 5a. ПОДКЛЮЧЕНИЕ К ПРОЦЕССУ (после перезапуска ядра) =====================
_pid, _state = current_process()
if _pid:
    print("найден живой процесс pid=%d (режим %s) по pid-файлу %s" % (_pid, (_state or {}).get("mode"), PID_FILE))
    monitor()
else:
    status()

In [ ]:
# ===================== 5b. ОСТАНОВКА ПРОЦЕССА (раскомментируйте вызов) =====================
# stop_core()
status()

## 6. Сводка

Таблица точности (доля верных ответов под фильтром `flexible-extract`, как в `eval/utils.py::parse_results`) с 95% доверительным интервалом (percentile bootstrap по задачам, `N_BOOTSTRAP` повторов) для каждой комбинации `remasking` x число шагов; дополнительно `strict-match`. Считает процесс из venv (`--mode summary`, модель не загружается), результат пишется в `summary.json`. Сравнений с другими экспериментами здесь нет.

In [ ]:
# ===================== 6. СВОДКА (процесс из venv) =====================
_pid, _state = current_process()
assert _pid is None, "Рабочий процесс ещё идёт (pid %d): дождитесь окончания или stop_core()" % _pid
write_run_files()
_res = subprocess.run(["env"] + PROCESS_ENV + [VENV_PYTHON] + PROCESS_PYTHON_FLAGS + ["exp1_core.py", "--mode", "summary"], cwd=WORK_DIR,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True, timeout=900)
print(_res.stdout[-6000:])
assert _res.returncode == 0, "сводка завершилась с ошибкой (см. вывод выше)"

In [ ]:
# ===================== 6a. (НЕОБЯЗАТЕЛЬНО) ЦИФРЫ ИЗ СТАТЬИ - ТОЛЬКО РУЧНОЙ ВВОД =====================
# Блокнот ничего не подставляет сам. Если нужно, впишите цифры из таблицы статьи вручную, в процентах.
# Формат ключа: (remasking, num_steps) -> число, например: {("low_confidence", 64): <число из статьи>}
PAPER_NUMBERS = {}

if not PAPER_NUMBERS:
    print("PAPER_NUMBERS пуст - вывод пропущен (так и задумано).")
else:
    _summary = json.loads(Path(OUT_DIR, "summary.json").read_text(encoding="utf-8"))
    ours = dict(((s["remasking"], s["num_steps"]), 100.0 * s["accuracy"]) for s in _summary["rows"])
    print("ВНИМАНИЕ: цифры статьи получены в других условиях (в статье длина генерации 512 токенов и полный test GSM8K, а не первые %d задач; здесь GEN_LENGTH=%d)." % (N_TASKS, GEN_LENGTH))
    print("Также НЕ ПРОВЕРЕНО, на каком коде посчитан margin в статье (оригинальный харнесс без HMM поддерживает только low_confidence и random).")
    print("%-16s %6s %20s %32s" % ("remasking", "steps", "наш результат, %", "из статьи, % (ввод вручную)"))
    for key, paper_value in PAPER_NUMBERS.items():
        mine = ours.get(key)
        print("%-16s %6d %20s %32s" % (key[0], key[1], "нет" if mine is None else "%.1f" % mine, paper_value))

In [ ]:
# ===================== 7. ФАКТЫ ДЛЯ ОТЧЁТА И НАПОМИНАНИЕ =====================
def _load(name):
    try:
        return json.loads(Path(OUT_DIR, name).read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return None


_env, _tasks, _smoke = _load("env.json"), _load("tasks.json"), _load("smoke.json")
print("pyjuice:", (_env or {}).get("pyjuice_mode"), "(в блокнотах 2-5 заглушка ЗАПРЕЩЕНА)")
print("HMM_CKPT:", HMM_CKPT, "(изменён по решению автора; в exp1 не используется)")
print("GEN_LENGTH:", GEN_LENGTH, "| DTYPE:", DTYPE, "| GPU:", (_env or {}).get("gpu_name"), (_env or {}).get("gpu_capability"))
print("Сверка кода модели (sha256):", dict((k, v.get("verdict")) for k, v in ((_env or {}).get("model_code_check") or {}).items()))
print("sha256 списка id задач:", (_tasks or {}).get("ids_sha256"))
print("sha256 общих параметров:", sha256_json(COMMON_PARAMS))
print("дымовой тест:", ("пройден " + str(_smoke.get("finished_utc"))) if _smoke else "не проводился")
print("Результаты:", OUT_DIR)
if os.path.isdir(OUT_DIR):
    for _name in sorted(os.listdir(OUT_DIR)):
        print("   ", _name)
print()
print("!" * 100)
print("НАПОМИНАНИЕ: этот блокнот НЕ может остановить машину. После окончания работы вручную остановите JupyterLab и ВМ")
print("в интерфейсе Yandex DataSphere, иначе продолжится тарификация GPU. Точные пункты меню НЕ ПРОВЕРЕНЫ.")
print("!" * 100)